# NIR Feed AI Model — Step-by-Step MVP

**Your role:** ML Model + AI Architecture + NIR dataset.

This notebook builds the first real ML layer of the project:

**NIR spectrum → preprocessing → PLS regression → CP/NDF/ADF/IVDMD prediction → evaluation → saved model**

The selected public dataset contains NIR absorbance spectra and laboratory reference measurements for Urochloa humidicola forage.

> **Scientific boundary:** this is a proof-of-concept calibration model. It is not yet a universal cattle-feed model and should not be used for real feeding decisions. The final system needs locally collected Indian feed/silage samples with laboratory reference values.

## 1. Install and import the libraries

You do not need advanced NumPy/Pandas knowledge to start. For this project, focus on arrays, DataFrames, filtering, missing values and basic statistics.

In [ ]:
# If needed, run this once in a terminal:
# pip install -r requirements.txt

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nir_feed_model import (
    load_tab_file,
    detect_targets,
    detect_wavelength_columns,
    prepare_target,
    train_target_model,
    tune_pls_components,
    fit_anomaly_detector,
    save_model,
)

ModuleNotFoundError: No module named 'nir_feed_model'

In [ ]:
from google.colab import files

uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("NIR_Feed_AI_Model_MVP.zip")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("nir_feed_project")

print("✅ Project extracted successfully!")

In [ ]:
!ls nir_feed_project

In [ ]:
import sys

sys.path.insert(0, "/content/nir_feed_project")

print("✅ ML project folder connected!")

In [ ]:
from nir_feed_model import (
    load_tab_file,
    detect_targets,
    detect_wavelength_columns,
    prepare_target,
    train_target_model,
    tune_pls_components,
    fit_anomaly_detector,
    save_model,
)

print("✅ ML model imported successfully!")

## 2. Put the dataset in this folder

Download the dataset from Harvard Dataverse:

**DOI:** `10.7910/DVN/XPNIQY`

Then place the `.tab` file beside this notebook.

The paper reports 1,112 samples, 1,050 spectral variables from 400–2498 nm at 2 nm intervals, with laboratory reference measurements for NDF, ADF, IVDMD and CP.

In [ ]:
from google.colab import files

uploaded_dataset = files.upload()

In [ ]:
DATA_DIR = Path(".")
tab_files = list(DATA_DIR.glob("*.tab"))

print("TAB files found:", tab_files)

if not tab_files:
    print("No .tab file found yet.")
    print("Download the CIAT/Urochloa humidicola dataset and place the .tab file here.")
else:
    DATA_PATH = tab_files[0]
    df = load_tab_file(DATA_PATH)
    print("Shape:", df.shape)
    display(df.head())

In [ ]:
from pathlib import Path
import pandas as pd

tab_files = list(Path("/content").glob("*.tab"))

print("TAB files found:")
for f in tab_files:
    print(f)

if tab_files:
    test_file = tab_files[0]

    print("\nUsing file:", test_file)
    print("File size:", test_file.stat().st_size / (1024 * 1024), "MB")

    # Read just the first 5 rows
    test_df = pd.read_csv(
        test_file,
        sep="\t",
        nrows=5
    )

    print("\n✅ Dataset read successfully!")
    print("Number of columns:", len(test_df.columns))
    print("\nFirst 20 column names:")
    print(test_df.columns[:20].tolist())

    display(test_df.head())
else:
    print("❌ No .tab file found.")

In [ ]:
# STEP 8 — Prepare the NIR data for Crude Protein prediction

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Use the file we successfully inspected
DATA_PATH = "/content/02a._Fulldata[1].tab"

# Load the complete dataset
df = pd.read_csv(DATA_PATH, sep="\t")

# Identify NIR wavelength columns
wavelength_cols = [
    col for col in df.columns
    if str(col).isdigit() and 400 <= int(col) <= 2498
]

# Sort wavelengths numerically
wavelength_cols = sorted(wavelength_cols, key=lambda x: int(x))

print("Number of NIR wavelengths:", len(wavelength_cols))
print("First wavelengths:", wavelength_cols[:10])
print("Last wavelengths:", wavelength_cols[-10:])

# PC = Crude Protein
target_col = "PC"

# Keep only rows where protein is available
model_df = df[wavelength_cols + [target_col]].copy()

# Convert everything to numeric
model_df = model_df.apply(pd.to_numeric, errors="coerce")

# Remove rows with missing values
model_df = model_df.dropna()

# X = NIR spectra
X = model_df[wavelength_cols].values

# y = laboratory Crude Protein
y = model_df[target_col].values

print("\n✅ Data prepared!")
print("Samples:", X.shape[0])
print("NIR features:", X.shape[1])
print("Target:", target_col)
print("Protein range:", y.min(), "to", y.max())

In [ ]:
# STEP 9 — Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("NIR features:", X_train.shape[1])

In [ ]:
# STEP 10 — First PLS Regression model

pls_model = PLSRegression(
    n_components=10,
    scale=True,
    max_iter=2000
)

pls_model.fit(X_train, y_train)

print("✅ PLS model trained successfully!")

In [ ]:
# STEP 11 — Evaluate the model

y_pred = pls_model.predict(X_test).ravel()

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("===== CRUDE PROTEIN MODEL =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

In [ ]:
# STEP 12 — Visualize actual vs predicted protein

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))

plt.scatter(y_test, y_pred, alpha=0.7)

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Laboratory Crude Protein")
plt.ylabel("Predicted Crude Protein")
plt.title("NIR PLS Model — Crude Protein")

plt.grid(True)
plt.show()

In [ ]:
# STEP 13 — Find the best number of PLS components using cross-validation

from sklearn.model_selection import KFold, cross_val_score
from sklearn.cross_decomposition import PLSRegression
import numpy as np
import matplotlib.pyplot as plt

# Cross-validation strategy
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Test different numbers of PLS components
component_range = range(2, 31)

cv_r2 = []
cv_rmse = []

for n in component_range:

    model = PLSRegression(
        n_components=n,
        scale=True,
        max_iter=2000
    )

    # R²
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="r2"
    )

    cv_r2.append(scores.mean())

    # RMSE
    rmse_scores = np.sqrt(
        -cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="neg_mean_squared_error"
        )
    )

    cv_rmse.append(rmse_scores.mean())

# Best component count
best_index = np.argmax(cv_r2)
best_components = list(component_range)[best_index]

print("====================================")
print("PLS COMPONENT OPTIMIZATION")
print("====================================")
print("Best number of components:", best_components)
print("Best CV R²:", round(cv_r2[best_index], 4))
print("Corresponding CV RMSE:", round(cv_rmse[best_index], 4))

In [ ]:
# STEP 13B — Plot component optimization

plt.figure(figsize=(8, 5))

plt.plot(
    list(component_range),
    cv_r2,
    marker="o"
)

plt.axvline(
    best_components,
    linestyle="--"
)

plt.xlabel("Number of PLS Components")
plt.ylabel("5-Fold Cross-Validation R²")
plt.title("PLS Component Optimization")

plt.grid(True)
plt.show()

In [ ]:
# STEP 13C — Inspect the best PLS component region

results = pd.DataFrame({
    "Components": list(component_range),
    "CV_R2": cv_r2,
    "CV_RMSE": cv_rmse
})

print(
    results[
        results["Components"].between(14, 24)
    ].to_string(index=False)
)

In [ ]:
# STEP 14 — Train optimized Crude Protein model

BEST_COMPONENTS = 17

final_cp_model = PLSRegression(
    n_components=BEST_COMPONENTS,
    scale=True,
    max_iter=2000
)

final_cp_model.fit(X_train, y_train)

# Predict on the untouched test set
y_pred_final = final_cp_model.predict(X_test).ravel()

# Calculate final test metrics
final_mae = mean_absolute_error(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_r2 = r2_score(y_test, y_pred_final)

print("========================================")
print("OPTIMIZED CRUDE PROTEIN MODEL")
print("========================================")
print("PLS components:", BEST_COMPONENTS)
print(f"Test MAE  : {final_mae:.4f}")
print(f"Test RMSE : {final_rmse:.4f}")
print(f"Test R²   : {final_r2:.4f}")

In [ ]:
# STEP 15 — NIR Spectral Preprocessing
# We will compare:
# 1. Raw NIR
# 2. SNV
# 3. Savitzky-Golay
# 4. SNV + Savitzky-Golay

import numpy as np
from scipy.signal import savgol_filter

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.cross_decomposition import PLSRegression


# -----------------------------
# SNV TRANSFORMER
# -----------------------------
class SNVTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        mean = np.mean(X, axis=1, keepdims=True)
        std = np.std(X, axis=1, keepdims=True)

        return (X - mean) / (std + 1e-8)


# -----------------------------
# SAVITZKY-GOLAY TRANSFORMER
# -----------------------------
class SavitzkyGolayTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, window_length=11, polyorder=2, deriv=1):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        return savgol_filter(
            X,
            window_length=self.window_length,
            polyorder=self.polyorder,
            deriv=self.deriv,
            axis=1
        )


print("✅ Spectral preprocessing tools created successfully!")

In [ ]:
# STEP 16 — Compare NIR preprocessing methods

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {

    "Raw NIR": Pipeline([
        ("pls", PLSRegression(
            n_components=17,
            scale=True,
            max_iter=2000
        ))
    ]),

    "SNV": Pipeline([
        ("snv", SNVTransformer()),
        ("pls", PLSRegression(
            n_components=17,
            scale=True,
            max_iter=2000
        ))
    ]),

    "Savitzky-Golay": Pipeline([
        ("sg", SavitzkyGolayTransformer(
            window_length=11,
            polyorder=2,
            deriv=1
        )),
        ("pls", PLSRegression(
            n_components=17,
            scale=True,
            max_iter=2000
        ))
    ]),

    "SNV + Savitzky-Golay": Pipeline([
        ("snv", SNVTransformer()),
        ("sg", SavitzkyGolayTransformer(
            window_length=11,
            polyorder=2,
            deriv=1
        )),
        ("pls", PLSRegression(
            n_components=17,
            scale=True,
            max_iter=2000
        ))
    ])
}


results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="r2"
    )

    results.append({
        "Preprocessing": name,
        "CV_R2_Mean": scores.mean(),
        "CV_R2_STD": scores.std()
    })


results_df = pd.DataFrame(results)

display(results_df.sort_values(
    "CV_R2_Mean",
    ascending=False
))

In [ ]:
# STEP 12 — Upload the NDF dataset

from google.colab import files
from pathlib import Path

uploaded_ndf = files.upload()

print("Uploaded files:")
for name in uploaded_ndf.keys():
    print(" -", name)

In [ ]:
# STEP 13 — Load NDF dataset

ndf_file = Path("02b. NDF.tab")

if not ndf_file.exists():
    # Handle Colab's possible renamed filename
    tab_files = list(Path(".").glob("*NDF*.tab"))
    if tab_files:
        ndf_file = tab_files[0]

print("Using file:", ndf_file)

ndf_df = load_tab_file(ndf_file)

print("NDF dataset shape:", ndf_df.shape)

print("\nFirst 20 columns:")
print(ndf_df.columns[:20].tolist())

print("\nLast 10 columns:")
print(ndf_df.columns[-10:].tolist())

display(ndf_df.head())

In [ ]:
# STEP 14 — Detect NDF target and spectral columns

print("Possible target columns:")

target_candidates = [
    c for c in ndf_df.columns
    if "NDF" in str(c).upper()
]

print(target_candidates)

wavelength_cols_ndf = detect_wavelength_columns(ndf_df)

print("\nNumber of wavelength columns:", len(wavelength_cols_ndf))

print("First wavelengths:")
print(wavelength_cols_ndf[:10])

print("Last wavelengths:")
print(wavelength_cols_ndf[-10:])

In [ ]:
# STEP 15 — Prepare NDF target and spectral matrix

NDF_TARGET = "NDF"

X_ndf, y_ndf, wavelength_cols_ndf, ndf_target_col = prepare_target(
    ndf_df,
    NDF_TARGET
)

print("X shape:", X_ndf.shape)
print("y shape:", y_ndf.shape)

print("\nNDF statistics:")
print(pd.Series(y_ndf).describe())

In [ ]:
# STEP 16 — Split NDF data

from sklearn.model_selection import train_test_split

X_train_ndf, X_test_ndf, y_train_ndf, y_test_ndf = train_test_split(
    X_ndf,
    y_ndf,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train_ndf.shape[0])
print("Testing samples :", X_test_ndf.shape[0])
print("Number of wavelengths:", X_train_ndf.shape[1])

In [ ]:
# STEP 17 — Initial NDF PLS model

from sklearn.cross_decomposition import PLSRegression

pls_ndf = PLSRegression(
    n_components=10,
    scale=True,
    max_iter=2000
)

pls_ndf.fit(X_train_ndf, y_train_ndf)

print("✅ NDF PLS model trained successfully!")

In [ ]:
# STEP 18 — Evaluate initial NDF model

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred_ndf = pls_ndf.predict(X_test_ndf).ravel()

ndf_mae = mean_absolute_error(y_test_ndf, y_pred_ndf)
ndf_rmse = np.sqrt(mean_squared_error(y_test_ndf, y_pred_ndf))
ndf_r2 = r2_score(y_test_ndf, y_pred_ndf)

print("========== INITIAL NDF MODEL ==========")
print(f"MAE  : {ndf_mae:.4f}")
print(f"RMSE : {ndf_rmse:.4f}")
print(f"R²   : {ndf_r2:.4f}")

In [ ]:
# STEP 19 — NDF predicted vs laboratory values

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_ndf,
    y_pred_ndf,
    alpha=0.7
)

min_val = min(y_test_ndf.min(), y_pred_ndf.min())
max_val = max(y_test_ndf.max(), y_pred_ndf.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "--"
)

plt.xlabel("Laboratory NDF")
plt.ylabel("Predicted NDF")
plt.title("NIR PLS Model — NDF")

plt.grid(True)
plt.show()

In [ ]:
# STEP 20 — Optimize NDF PLS components

ndf_tuning = tune_pls_components(
    X_train_ndf,
    y_train_ndf,
    components=range(1, 31)
)

display(ndf_tuning)

In [ ]:
# STEP 17 — Train the optimized NDF PLS model

BEST_NDF_COMPONENTS = 18

final_ndf_model = PLSRegression(
    n_components=BEST_NDF_COMPONENTS,
    scale=True,
    max_iter=2000
)

final_ndf_model.fit(X_train_ndf, y_train_ndf)

print("✅ Final NDF PLS model trained!")
print("PLS components:", BEST_NDF_COMPONENTS)

In [ ]:
# STEP 18 — Evaluate the final NDF model

y_pred_ndf = final_ndf_model.predict(X_test_ndf).ravel()

ndf_mae = mean_absolute_error(y_test_ndf, y_pred_ndf)
ndf_rmse = np.sqrt(mean_squared_error(y_test_ndf, y_pred_ndf))
ndf_r2 = r2_score(y_test_ndf, y_pred_ndf)

print("========== NDF MODEL EVALUATION ==========")
print(f"MAE  : {ndf_mae:.4f}")
print(f"RMSE : {ndf_rmse:.4f}")
print(f"R²   : {ndf_r2:.4f}")

In [ ]:
# STEP 19 — Laboratory NDF vs Predicted NDF

plt.figure(figsize=(8, 6))

plt.scatter(y_test_ndf, y_pred_ndf, alpha=0.7)

# Perfect prediction line
min_val = min(y_test_ndf.min(), y_pred_ndf.min())
max_val = max(y_test_ndf.max(), y_pred_ndf.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Laboratory NDF")
plt.ylabel("Predicted NDF")
plt.title("NIR PLS Model — NDF")

plt.grid(True)
plt.show()

In [ ]:
!ls /content/*.tab

In [ ]:
# STEP 20A — Load ADF dataset

ADF_PATH = Path("/content/02c._ADF[1].tab")

adf_df = load_tab_file(ADF_PATH)

print("✅ ADF dataset loaded!")
print("Shape:", adf_df.shape)
print(adf_df.head())

In [ ]:
# STEP 20 — Prepare ADF target and spectral matrix

ADF_TARGET = "ADF"

X_adf, y_adf, wavelength_cols_adf, adf_target_col = prepare_target(
    adf_df,
    ADF_TARGET
)

print("X shape:", X_adf.shape)
print("y shape:", y_adf.shape)

print("\nADF statistics:")
print(pd.Series(y_adf).describe())

In [ ]:
# STEP 20A — Load ADF dataset

ADF_PATH = Path("/content/02c._ADF[1].tab")

adf_df = load_tab_file(ADF_PATH)

print("✅ ADF dataset loaded!")
print("Shape:", adf_df.shape)
print(adf_df.head())

In [ ]:
# STEP 21 — Train/Test split for ADF

X_train_adf, X_test_adf, y_train_adf, y_test_adf = train_test_split(
    X_adf,
    y_adf,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train_adf.shape[0])
print("Test samples:", X_test_adf.shape[0])
print("Number of wavelengths:", X_train_adf.shape[1])

In [ ]:
# STEP 22 — Optimize PLS components for ADF

component_results_adf = []

for n in range(2, 31):
    pls = PLSRegression(
        n_components=n,
        scale=True,
        max_iter=2000
    )

    scores = cross_val_score(
        pls,
        X_train_adf,
        y_train_adf,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )

    component_results_adf.append({
        "n_components": n,
        "CV_RMSE_mean": -scores.mean(),
        "CV_RMSE_std": scores.std()
    })

adf_components_df = pd.DataFrame(component_results_adf)

display(
    adf_components_df.sort_values(
        "CV_RMSE_mean",
        ascending=True
    )
)

In [ ]:
# STEP — Train the optimized ADF PLS model

BEST_ADF_COMPONENTS = 21

final_adf_model = PLSRegression(
    n_components=BEST_ADF_COMPONENTS,
    scale=True,
    max_iter=2000
)

final_adf_model.fit(X_train_adf, y_train_adf)

print("✅ Final ADF PLS model trained!")
print("PLS components:", BEST_ADF_COMPONENTS)

In [ ]:
# STEP — Evaluate final ADF model on untouched test set

y_pred_adf = final_adf_model.predict(X_test_adf).ravel()

adf_mae = mean_absolute_error(y_test_adf, y_pred_adf)
adf_rmse = np.sqrt(mean_squared_error(y_test_adf, y_pred_adf))
adf_r2 = r2_score(y_test_adf, y_pred_adf)

print("======================================")
print("OPTIMIZED ADF PLS MODEL")
print("======================================")
print("PLS components:", BEST_ADF_COMPONENTS)
print(f"Test MAE  : {adf_mae:.4f}")
print(f"Test RMSE : {adf_rmse:.4f}")
print(f"Test R²   : {adf_r2:.4f}")

In [ ]:
# STEP — ADF Predicted vs Laboratory Values

plt.figure(figsize=(8, 7))

plt.scatter(
    y_test_adf,
    y_pred_adf,
    alpha=0.7
)

# Perfect prediction line
min_val = min(y_test_adf.min(), y_pred_adf.min())
max_val = max(y_test_adf.max(), y_pred_adf.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Laboratory ADF")
plt.ylabel("Predicted ADF")
plt.title("NIR PLS Model — ADF")

plt.grid(True)
plt.show()

In [ ]:
# STEP — Prepare IVDMD target and spectral matrix

IVDMD_TARGET = "IVDMD"

X_ivdmd, y_ivdmd = prepare_target(
    ivdmd_df,
    IVDMD_TARGET
)

print("X shape:", X_ivdmd.shape)
print("y shape:", y_ivdmd.shape)

print("\nIVDMD statistics:")
print(pd.Series(y_ivdmd).describe())

In [ ]:
!ls -lh /content/*.tab


In [ ]:
# STEP 21 — Prepare IVDMD target correctly

IVDMD_TARGET = "DIVMS"

# Get wavelength columns directly
wavelength_cols_ivdmd = [
    col for col in df.columns
    if str(col).isdigit()
]

# Keep only rows where IVDMD (DIVMS) exists
ivdmd_mask = pd.to_numeric(df[IVDMD_TARGET], errors="coerce").notna()

# Spectral data
x_ivdmd = df.loc[ivdmd_mask, wavelength_cols_ivdmd].apply(
    pd.to_numeric, errors="coerce"
).values

# Target
y_ivdmd = pd.to_numeric(
    df.loc[ivdmd_mask, IVDMD_TARGET],
    errors="coerce"
).values

# Remove rows with any missing spectral values
valid_mask = ~np.isnan(x_ivdmd).any(axis=1)

x_ivdmd = x_ivdmd[valid_mask]
y_ivdmd = y_ivdmd[valid_mask]

print("✅ IVDMD data prepared successfully!")
print("X shape:", x_ivdmd.shape)
print("y shape:", y_ivdmd.shape)

print("\nIVDMD statistics:")
print(pd.Series(y_ivdmd).describe())

In [ ]:
# STEP 22 — Split IVDMD data into training and test sets

from sklearn.model_selection import train_test_split

X_train_ivdmd, X_test_ivdmd, y_train_ivdmd, y_test_ivdmd = train_test_split(
    x_ivdmd,
    y_ivdmd,
    test_size=0.20,
    random_state=42
)

print("✅ IVDMD train/test split complete!")
print("Training samples:", X_train_ivdmd.shape[0])
print("Test samples:", X_test_ivdmd.shape[0])
print("Number of wavelengths:", X_train_ivdmd.shape[1])

In [ ]:
# STEP 23 — Find the optimal number of PLS components for IVDMD

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
import pandas as pd

max_components_ivdmd = 30

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

ivdmd_results = []

for n in range(2, max_components_ivdmd + 1):

    pls = PLSRegression(
        n_components=n,
        scale=True,
        max_iter=2000
    )

    scores = cross_val_score(
        pls,
        X_train_ivdmd,
        y_train_ivdmd,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    rmse_scores = -scores

    ivdmd_results.append({
        "n_components": n,
        "CV_RMSE_mean": rmse_scores.mean(),
        "CV_RMSE_std": rmse_scores.std()
    })

ivdmd_results_df = pd.DataFrame(ivdmd_results)

# Sort by lowest CV RMSE
ivdmd_results_df = ivdmd_results_df.sort_values(
    "CV_RMSE_mean",
    ascending=True
).reset_index(drop=True)

display(ivdmd_results_df)

In [ ]:
# STEP 24 — Train the optimized IVDMD PLS model

BEST_IVDMD_COMPONENTS = 21

final_ivdmd_model = PLSRegression(
    n_components=BEST_IVDMD_COMPONENTS,
    scale=True,
    max_iter=2000
)

final_ivdmd_model.fit(
    X_train_ivdmd,
    y_train_ivdmd
)

print("✅ Final IVDMD PLS model trained!")
print("PLS components:", BEST_IVDMD_COMPONENTS)

In [ ]:
# STEP 25 — Evaluate the optimized IVDMD PLS model

y_pred_ivdmd = final_ivdmd_model.predict(
    X_test_ivdmd
).ravel()

ivdmd_mae = mean_absolute_error(
    y_test_ivdmd,
    y_pred_ivdmd
)

ivdmd_rmse = np.sqrt(
    mean_squared_error(
        y_test_ivdmd,
        y_pred_ivdmd
    )
)

ivdmd_r2 = r2_score(
    y_test_ivdmd,
    y_pred_ivdmd
)

print("======================================")
print("OPTIMIZED IVDMD PLS MODEL")
print("======================================")
print("PLS components:", BEST_IVDMD_COMPONENTS)
print(f"Test MAE  : {ivdmd_mae:.4f}")
print(f"Test RMSE : {ivdmd_rmse:.4f}")
print(f"Test R²   : {ivdmd_r2:.4f}")

In [ ]:
# STEP 26 — Diagnose unusual IVDMD values

print("======================================")
print("IVDMD QUALITY CHECK")
print("======================================")

# Values above 100
high_ivdmd = df[
    pd.to_numeric(df["DIVMS"], errors="coerce") > 100
][["Country", "Location", "Harvest date", "DIVMS"]]

print("\nIVDMD values > 100:")
display(high_ivdmd)

print("\nNumber of values > 100:", len(high_ivdmd))

# Values below 0
low_ivdmd = df[
    pd.to_numeric(df["DIVMS"], errors="coerce") < 0
][["Country", "Location", "Harvest date", "DIVMS"]]

print("\nNumber of values < 0:", len(low_ivdmd))

# Distribution
print("\nIVDMD percentiles:")
print(
    pd.to_numeric(df["DIVMS"], errors="coerce")
    .describe(percentiles=[0.01, 0.05, 0.95, 0.99])
)

In [ ]:
# STEP 27 — Inspect extreme IVDMD samples

ivdmd_numeric = pd.to_numeric(df["DIVMS"], errors="coerce")

extreme_ivdmd = df.loc[
    ivdmd_numeric.index[
        (ivdmd_numeric <= 5) | (ivdmd_numeric >= 80)
    ],
    ["Country", "Location", "Harvest date", "DIVMS"]
].copy()

extreme_ivdmd["DIVMS"] = pd.to_numeric(
    extreme_ivdmd["DIVMS"], errors="coerce"
)

extreme_ivdmd = extreme_ivdmd.sort_values("DIVMS")

print("Extreme IVDMD samples:")
display(extreme_ivdmd)

print("\nNumber of extreme samples:", len(extreme_ivdmd))

In [ ]:
# STEP 28 — Check extreme IVDMD values in train/test

print("Checking extreme IVDMD values...")
print("================================")

for value in [0.613, 108.53]:

    in_train = np.any(np.isclose(y_train, value))
    in_test = np.any(np.isclose(y_test, value))

    print(f"\nIVDMD = {value}")
    print("TRAIN:", in_train)
    print("TEST :", in_test)

In [ ]:
# STEP 29 — Check actual IVDMD values used by the model

print("======================================")
print("ACTUAL MODELING IVDMD CHECK")
print("======================================")

print("\nFull dataframe:")
print("Minimum:", pd.to_numeric(df["DIVMS"], errors="coerce").min())
print("Maximum:", pd.to_numeric(df["DIVMS"], errors="coerce").max())

print("\nModel target y:")
print("Minimum:", np.min(y))
print("Maximum:", np.max(y))
print("Number of samples:", len(y))

print("\nExtreme values inside y:")
print("Values <= 5:", y[y <= 5])
print("Values >= 80:", y[y >= 80])

In [ ]:
# STEP 30 — Rebuild clean IVDMD dataset

print("======================================")
print("REBUILDING IVDMD DATASET")
print("======================================")

# Identify NIR wavelength columns
wavelength_cols_ivdmd = [
    c for c in df.columns
    if str(c).isdigit() and 400 <= int(c) <= 2498
]

# Convert target to numeric
ivdmd_target = pd.to_numeric(df["DIVMS"], errors="coerce")

# Keep rows with valid IVDMD and complete NIR spectra
valid_ivdmd = (
    ivdmd_target.notna() &
    df[wavelength_cols_ivdmd].notna().all(axis=1)
)

# Build clean X and y
X_ivdmd = df.loc[valid_ivdmd, wavelength_cols_ivdmd].astype(float).values
y_ivdmd = ivdmd_target.loc[valid_ivdmd].values

print("\nX_ivdmd shape:", X_ivdmd.shape)
print("y_ivdmd shape:", y_ivdmd.shape)

print("\nIVDMD target statistics:")
print(pd.Series(y_ivdmd).describe())

print("\nValues <= 5:")
print(y_ivdmd[y_ivdmd <= 5])

print("\nValues >= 80:")
print(y_ivdmd[y_ivdmd >= 80])

In [ ]:
# STEP 31 — Clean IVDMD train/test split

from sklearn.model_selection import train_test_split

# Keep original dataframe indices so we can trace unusual samples
ivdmd_indices = df.index[valid_ivdmd].to_numpy()

# Split X, y, and original indices together
X_ivdmd_train, X_ivdmd_test, y_ivdmd_train, y_ivdmd_test, idx_ivdmd_train, idx_ivdmd_test = train_test_split(
    X_ivdmd,
    y_ivdmd,
    ivdmd_indices,
    test_size=0.20,
    random_state=42
)

print("======================================")
print("IVDMD TRAIN / TEST SPLIT")
print("======================================")

print("\nTraining samples:", len(y_ivdmd_train))
print("Test samples:", len(y_ivdmd_test))

print("\n108.53 location:")

if np.any(np.isclose(y_ivdmd_train, 108.53)):
    print("108.53 -> TRAIN")

elif np.any(np.isclose(y_ivdmd_test, 108.53)):
    print("108.53 -> TEST")

else:
    print("108.53 -> NOT FOUND")

print("\nTraining IVDMD range:")
print("Min:", np.min(y_ivdmd_train))
print("Max:", np.max(y_ivdmd_train))

print("\nTest IVDMD range:")
print("Min:", np.min(y_ivdmd_test))
print("Max:", np.max(y_ivdmd_test))

In [ ]:
# STEP 32 — Diagnose IVDMD prediction errors

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train IVDMD PLS model
pls_ivdmd = PLSRegression(n_components=21)

pls_ivdmd.fit(X_ivdmd_train, y_ivdmd_train)

# Predict test set
y_ivdmd_pred = pls_ivdmd.predict(X_ivdmd_test).ravel()

# Overall metrics
mae_ivdmd = mean_absolute_error(y_ivdmd_test, y_ivdmd_pred)
rmse_ivdmd = np.sqrt(mean_squared_error(y_ivdmd_test, y_ivdmd_pred))
r2_ivdmd = r2_score(y_ivdmd_test, y_ivdmd_pred)

print("======================================")
print("IVDMD BASELINE MODEL")
print("======================================")

print(f"\nMAE  : {mae_ivdmd:.4f}")
print(f"RMSE : {rmse_ivdmd:.4f}")
print(f"R²   : {r2_ivdmd:.4f}")
print(f"R² % : {r2_ivdmd * 100:.2f}%")

# Create error table
error_table = pd.DataFrame({
    "Actual": y_ivdmd_test,
    "Predicted": y_ivdmd_pred,
    "Absolute_Error": np.abs(y_ivdmd_test - y_ivdmd_pred),
    "Original_Index": idx_ivdmd_test
})

# Sort by largest error
error_table = error_table.sort_values(
    "Absolute_Error",
    ascending=False
)

print("\nTop 10 largest prediction errors:")
display(error_table.head(10))

In [ ]:
# STEP 33 — Measure impact of the extreme test sample

# Identify the 108.53 sample
normal_mask = ~np.isclose(y_ivdmd_test, 108.53)

y_test_without_extreme = y_ivdmd_test[normal_mask]
pred_without_extreme = y_ivdmd_pred[normal_mask]

mae_without = mean_absolute_error(
    y_test_without_extreme,
    pred_without_extreme
)

rmse_without = np.sqrt(
    mean_squared_error(
        y_test_without_extreme,
        pred_without_extreme
    )
)

r2_without = r2_score(
    y_test_without_extreme,
    pred_without_extreme
)

print("======================================")
print("IVDMD DIAGNOSTIC — WITHOUT 108.53")
print("======================================")

print(f"\nMAE  : {mae_without:.4f}")
print(f"RMSE : {rmse_without:.4f}")
print(f"R²   : {r2_without:.4f}")
print(f"R² % : {r2_without * 100:.2f}%")

print("\nComparison:")
print(f"Original R² : {r2_ivdmd * 100:.2f}%")
print(f"New R²      : {r2_without * 100:.2f}%")
print(f"Improvement : {(r2_without - r2_ivdmd) * 100:.2f} percentage points")

In [ ]:
# STEP 34 — Check IVDMD performance on the normal/in-range test set

# Training range
train_min = np.min(y_ivdmd_train)
train_max = np.max(y_ivdmd_train)

# Identify test samples inside training range
in_range_mask = (
    (y_ivdmd_test >= train_min) &
    (y_ivdmd_test <= train_max)
)

y_test_in_range = y_ivdmd_test[in_range_mask]
pred_in_range = y_ivdmd_pred[in_range_mask]

mae_in_range = mean_absolute_error(
    y_test_in_range,
    pred_in_range
)

rmse_in_range = np.sqrt(
    mean_squared_error(
        y_test_in_range,
        pred_in_range
    )
)

r2_in_range = r2_score(
    y_test_in_range,
    pred_in_range
)

print("======================================")
print("IVDMD IN-RANGE TEST PERFORMANCE")
print("======================================")

print("\nTraining range:")
print(f"{train_min:.3f} → {train_max:.3f}")

print("\nTest samples:")
print(len(y_ivdmd_test))

print("In-range test samples:")
print(len(y_test_in_range))

print("\nPerformance:")
print(f"MAE  : {mae_in_range:.4f}")
print(f"RMSE : {rmse_in_range:.4f}")
print(f"R²   : {r2_in_range:.4f}")
print(f"R² % : {r2_in_range * 100:.2f}%")

In [ ]:
# STEP 33B — IVDMD Actual vs Predicted Graph

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    y_ivdmd_test,
    y_ivdmd_pred,
    alpha=0.7
)

# Perfect prediction line
min_val = min(y_ivdmd_test.min(), y_ivdmd_pred.min())
max_val = max(y_ivdmd_test.max(), y_ivdmd_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Actual IVDMD (%)")
plt.ylabel("Predicted IVDMD (%)")
plt.title("IVDMD — Actual vs Predicted")

plt.text(
    0.05,
    0.95,
    f"R² = {r2_ivdmd:.3f}\nRMSE = {rmse_ivdmd:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 34B — IVDMD In-Range Actual vs Predicted Graph

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_in_range,
    pred_in_range,
    alpha=0.7
)

# Perfect prediction line
min_val = min(y_test_in_range.min(), pred_in_range.min())
max_val = max(y_test_in_range.max(), pred_in_range.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Actual IVDMD (%)")
plt.ylabel("Predicted IVDMD (%)")
plt.title("IVDMD — In-Range Actual vs Predicted")

plt.text(
    0.05,
    0.95,
    f"R² = {r2_in_range:.3f}\nRMSE = {rmse_in_range:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 35 — IVDMD preprocessing comparison
# IMPORTANT: test set is NOT used here.

from sklearn.model_selection import KFold, cross_val_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
import numpy as np
import pandas as pd

# Use only training data
X = X_ivdmd_train
y_train_local = y_ivdmd_train

# 5-fold CV
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

# ------------------------------------------------
# 1. RAW
# ------------------------------------------------
X_raw = X.copy()

pls = PLSRegression(n_components=21)

scores = cross_val_score(
    pls,
    X_raw,
    y_train_local,
    cv=cv,
    scoring="r2"
)

results.append({
    "Preprocessing": "Raw",
    "CV_R2_Mean": scores.mean(),
    "CV_R2_STD": scores.std()
})

# ------------------------------------------------
# 2. SNV
# ------------------------------------------------
row_mean = X.mean(axis=1, keepdims=True)
row_std = X.std(axis=1, keepdims=True)

X_snv = (X - row_mean) / (row_std + 1e-12)

pls = PLSRegression(n_components=21)

scores = cross_val_score(
    pls,
    X_snv,
    y_train_local,
    cv=cv,
    scoring="r2"
)

results.append({
    "Preprocessing": "SNV",
    "CV_R2_Mean": scores.mean(),
    "CV_R2_STD": scores.std()
})

# ------------------------------------------------
# 3. Savitzky-Golay
# ------------------------------------------------
X_sg = savgol_filter(
    X,
    window_length=15,
    polyorder=2,
    deriv=1,
    axis=1
)

pls = PLSRegression(n_components=21)

scores = cross_val_score(
    pls,
    X_sg,
    y_train_local,
    cv=cv,
    scoring="r2"
)

results.append({
    "Preprocessing": "Savitzky-Golay 1st derivative",
    "CV_R2_Mean": scores.mean(),
    "CV_R2_STD": scores.std()
})

# ------------------------------------------------
# 4. SNV + Savitzky-Golay
# ------------------------------------------------
row_mean = X_sg.mean(axis=1, keepdims=True)
row_std = X_sg.std(axis=1, keepdims=True)

X_snv_sg = (X_sg - row_mean) / (row_std + 1e-12)

pls = PLSRegression(n_components=21)

scores = cross_val_score(
    pls,
    X_snv_sg,
    y_train_local,
    cv=cv,
    scoring="r2"
)

results.append({
    "Preprocessing": "SNV + Savitzky-Golay",
    "CV_R2_Mean": scores.mean(),
    "CV_R2_STD": scores.std()
})

# ------------------------------------------------
# Results
# ------------------------------------------------
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "CV_R2_Mean",
    ascending=False
).reset_index(drop=True)

print("======================================")
print("IVDMD PREPROCESSING COMPARISON")
print("======================================")

display(results_df)

In [ ]:
# STEP 36 — Optimize IVDMD PLS components using RAW NIR

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
import numpy as np

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

component_results = []

# Test 2 to 35 components
for n in range(2, 36):

    pls = PLSRegression(
        n_components=n
    )

    scores = cross_val_score(
        pls,
        X_ivdmd_train,
        y_ivdmd_train,
        cv=cv,
        scoring="r2"
    )

    component_results.append({
        "Components": n,
        "CV_R2_Mean": scores.mean(),
        "CV_R2_STD": scores.std(),
        "CV_RMSE": np.sqrt(
            -cross_val_score(
                pls,
                X_ivdmd_train,
                y_ivdmd_train,
                cv=cv,
                scoring="neg_mean_squared_error"
            ).mean()
        )
    })

components_df = pd.DataFrame(component_results)

# Sort by CV R²
best_components = components_df.sort_values(
    "CV_R2_Mean",
    ascending=False
).reset_index(drop=True)

print("======================================")
print("IVDMD PLS COMPONENT OPTIMIZATION")
print("======================================")

display(best_components.head(10))

print("\nBEST NUMBER OF COMPONENTS:")
print(best_components.iloc[0])

In [ ]:
# STEP 37 — Compare IVDMD PLS vs SVR
# Test set is NOT used.

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("======================================")
print("IVDMD — PLS vs SVR")
print("======================================")

# -----------------------------
# PLS baseline
# -----------------------------
pls_model = PLSRegression(n_components=21)

pls_scores = cross_val_score(
    pls_model,
    X_ivdmd_train,
    y_ivdmd_train,
    cv=cv,
    scoring="r2"
)

print("\nPLS (21 components)")
print(f"CV R² : {pls_scores.mean():.4f}")
print(f"CV R² % : {pls_scores.mean() * 100:.2f}%")

# -----------------------------
# SVR RBF
# -----------------------------
svr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        C=100,
        gamma="scale",
        epsilon=0.1
    ))
])

svr_scores = cross_val_score(
    svr_model,
    X_ivdmd_train,
    y_ivdmd_train,
    cv=cv,
    scoring="r2"
)

print("\nSVR (RBF)")
print(f"CV R² : {svr_scores.mean():.4f}")
print(f"CV R² % : {svr_scores.mean() * 100:.2f}%")

print("\nDifference:")
print(
    f"{(svr_scores.mean() - pls_scores.mean()) * 100:.2f}"
    " percentage points"
)

In [ ]:
# STEP 38 — IVDMD wavelength-region comparison
# Test set remains untouched.

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
import pandas as pd

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Wavelengths corresponding to the columns
wavelengths = np.array([
    int(c) for c in wavelength_cols_ivdmd
])

regions = {
    "Full 400-2498 nm": (400, 2498),
    "450-2400 nm": (450, 2400),
    "500-2300 nm": (500, 2300),
    "600-2200 nm": (600, 2200),
    "800-2000 nm": (800, 2000)
}

region_results = []

for name, (low, high) in regions.items():

    mask = (
        (wavelengths >= low) &
        (wavelengths <= high)
    )

    X_region = X_ivdmd_train[:, mask]

    pls = PLSRegression(
        n_components=21
    )

    scores = cross_val_score(
        pls,
        X_region,
        y_ivdmd_train,
        cv=cv,
        scoring="r2"
    )

    region_results.append({
        "Region": name,
        "Wavelengths": X_region.shape[1],
        "CV_R2_Mean": scores.mean(),
        "CV_R2_STD": scores.std()
    })

region_results_df = pd.DataFrame(region_results)

region_results_df = region_results_df.sort_values(
    "CV_R2_Mean",
    ascending=False
).reset_index(drop=True)

print("======================================")
print("IVDMD WAVELENGTH REGION COMPARISON")
print("======================================")

display(region_results_df)

In [ ]:
# STEP 39 — Final IVDMD PLS model

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Final selected model
final_ivdmd_model = PLSRegression(
    n_components=21
)

# Train on training data only
final_ivdmd_model.fit(
    X_ivdmd_train,
    y_ivdmd_train
)

# Predict untouched test set
final_ivdmd_pred = final_ivdmd_model.predict(
    X_ivdmd_test
).ravel()

# Metrics
final_mae = mean_absolute_error(
    y_ivdmd_test,
    final_ivdmd_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_ivdmd_test,
        final_ivdmd_pred
    )
)

final_r2 = r2_score(
    y_ivdmd_test,
    final_ivdmd_pred
)

print("======================================")
print("FINAL IVDMD PLS MODEL")
print("======================================")

print("\nModel:")
print("Algorithm: PLS Regression")
print("Components: 21")
print("Spectral range: 400–2498 nm")
print("Preprocessing: Raw NIR")

print("\nTest Performance:")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"R²   : {final_r2:.4f}")
print(f"R² % : {final_r2 * 100:.2f}%")

# -----------------------------------
# Graph
# -----------------------------------

plt.figure(figsize=(8, 6))

plt.scatter(
    y_ivdmd_test,
    final_ivdmd_pred,
    alpha=0.7
)

min_val = min(
    y_ivdmd_test.min(),
    final_ivdmd_pred.min()
)

max_val = max(
    y_ivdmd_test.max(),
    final_ivdmd_pred.max()
)

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("Actual IVDMD (%)")
plt.ylabel("Predicted IVDMD (%)")
plt.title("Final IVDMD PLS Model — Actual vs Predicted")

plt.text(
    0.05,
    0.95,
    f"R² = {final_r2:.3f}\nRMSE = {final_rmse:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 40 — Robust IVDMD repeated cross-validation

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.cross_decomposition import PLSRegression
import numpy as np

print("======================================")
print("IVDMD ROBUST CROSS-VALIDATION")
print("======================================")

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

pls_robust = PLSRegression(
    n_components=21
)

cv_scores = cross_val_score(
    pls_robust,
    X_ivdmd,
    y_ivdmd,
    cv=rkf,
    scoring="r2"
)

print("\nNumber of evaluations:", len(cv_scores))

print(f"\nMean R² : {cv_scores.mean():.4f}")
print(f"Mean R² % : {cv_scores.mean() * 100:.2f}%")

print(f"\nStd R²  : {cv_scores.std():.4f}")

print(f"\nMinimum R² : {cv_scores.min():.4f}")
print(f"Maximum R² : {cv_scores.max():.4f}")

print("\nApproximate 95% range:")
print(
    f"{(cv_scores.mean() - 1.96 * cv_scores.std()):.4f}"
    " to "
    f"{(cv_scores.mean() + 1.96 * cv_scores.std()):.4f}"
)

In [ ]:
# STEP 41 — Check repeated-CV sensitivity to the 108.53 sample

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
import numpy as np

print("======================================")
print("IVDMD EXTREME-SAMPLE CV DIAGNOSTIC")
print("======================================")

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

model = PLSRegression(
    n_components=21
)

cv_pred = cross_val_predict(
    model,
    X_ivdmd,
    y_ivdmd,
    cv=kf
).ravel()

# Overall CV R²
overall_r2 = r2_score(
    y_ivdmd,
    cv_pred
)

print(f"\nOverall CV R²: {overall_r2:.4f}")
print(f"Overall CV R² %: {overall_r2 * 100:.2f}%")

# Locate 108.53
extreme_mask = np.isclose(
    y_ivdmd,
    108.53,
    atol=0.001
)

print("\n108.53 sample found:", extreme_mask.sum())

if extreme_mask.sum() > 0:

    actual_extreme = y_ivdmd[extreme_mask][0]
    pred_extreme = cv_pred[extreme_mask][0]

    print(f"Actual   : {actual_extreme:.3f}")
    print(f"Predicted: {pred_extreme:.3f}")
    print(f"Error    : {abs(actual_extreme - pred_extreme):.3f}")

# CV R² excluding ONLY the extreme sample
normal_mask = ~extreme_mask

r2_without_extreme = r2_score(
    y_ivdmd[normal_mask],
    cv_pred[normal_mask]
)

print("\nCV R² without 108.53:")
print(f"{r2_without_extreme:.4f}")
print(f"{r2_without_extreme * 100:.2f}%")

In [ ]:
# STEP 42 — Combined Nutritional AI Model Results

import pandas as pd

model_results = pd.DataFrame([
    {
        "Target": "Crude Protein (CP)",
        "Model": "PLS Regression",
        "Components": 17,
        "Spectral Range": "400–2498 nm",
        "Preprocessing": "Raw NIR",
        "Test_R2": 0.9441,
        "Test_R2_Percent": 94.41,
        "Test_MAE": 0.4685,
        "Test_RMSE": 0.6129
    },
    {
        "Target": "NDF",
        "Model": "PLS Regression",
        "Components": 18,
        "Spectral Range": "400–2498 nm",
        "Preprocessing": "Raw NIR",
        "Test_R2": 0.9401,
        "Test_R2_Percent": 94.01,
        "Test_MAE": None,
        "Test_RMSE": None
    },
    {
        "Target": "ADF",
        "Model": "PLS Regression",
        "Components": 21,
        "Spectral Range": "400–2498 nm",
        "Preprocessing": "Raw NIR",
        "Test_R2": 0.9430,
        "Test_R2_Percent": 94.30,
        "Test_MAE": 0.8613,
        "Test_RMSE": 1.2285
    },
    {
        "Target": "IVDMD",
        "Model": "PLS Regression",
        "Components": 21,
        "Spectral Range": "400–2498 nm",
        "Preprocessing": "Raw NIR",
        "Test_R2": 0.8316,
        "Test_R2_Percent": 83.16,
        "Test_MAE": 2.0778,
        "Test_RMSE": 4.0120
    }
])

print("==============================================")
print("NIR FEED NUTRITIONAL AI — MODEL PERFORMANCE")
print("==============================================")

display(model_results)

print("\nAverage Test R²:")
print(
    f"{model_results['Test_R2_Percent'].mean():.2f}%"
)

In [ ]:
# STEP 43 — Feed Quality & Farmer Advisory Engine
#
# IMPORTANT:
# These thresholds are PROVISIONAL prototype thresholds.
# They are NOT laboratory safety limits or ration-formulation rules.
# We will validate/refine them later with domain literature/expert input.

def assess_feed_quality(cp, ndf, adf, ivdmd):

    flags = []
    recommendations = []

    # =========================================================
    # 1. CRUDE PROTEIN
    # =========================================================

    if cp < 7:
        cp_status = "LOW"
        flags.append("Low crude protein")
        recommendations.append(
            "Consider supplementing the ration with a suitable protein source."
        )

    elif cp < 12:
        cp_status = "MODERATE"

    else:
        cp_status = "GOOD"

    # =========================================================
    # 2. NDF
    # =========================================================

    if ndf > 60:
        ndf_status = "HIGH"
        flags.append("High fiber (NDF)")
        recommendations.append(
            "High NDF may reduce voluntary feed intake; review forage quality."
        )

    elif ndf > 40:
        ndf_status = "MODERATE"

    else:
        ndf_status = "LOW"

    # =========================================================
    # 3. ADF
    # =========================================================

    if adf > 40:
        adf_status = "HIGH"
        flags.append("High ADF")
        recommendations.append(
            "High ADF may indicate lower digestibility; review feed maturity and quality."
        )

    elif adf > 30:
        adf_status = "MODERATE"

    else:
        adf_status = "LOW"

    # =========================================================
    # 4. IVDMD
    # =========================================================

    if ivdmd < 55:
        ivdmd_status = "LOW"
        flags.append("Low estimated digestibility")
        recommendations.append(
            "Low predicted digestibility; consider checking forage maturity, processing and storage quality."
        )

    elif ivdmd < 65:
        ivdmd_status = "MODERATE"

    else:
        ivdmd_status = "GOOD"

    # =========================================================
    # 5. OVERALL STATUS
    # =========================================================

    serious_flags = 0

    if cp_status == "LOW":
        serious_flags += 1

    if ndf_status == "HIGH":
        serious_flags += 1

    if adf_status == "HIGH":
        serious_flags += 1

    if ivdmd_status == "LOW":
        serious_flags += 1

    if serious_flags >= 2:
        overall_status = "NEEDS ATTENTION"

    elif serious_flags == 1:
        overall_status = "MODERATE"

    else:
        overall_status = "GOOD"

    # =========================================================
    # 6. FINAL RESULT
    # =========================================================

    result = {
        "Overall Status": overall_status,

        "Crude Protein": {
            "Value (%)": round(cp, 2),
            "Status": cp_status
        },

        "NDF": {
            "Value (%)": round(ndf, 2),
            "Status": ndf_status
        },

        "ADF": {
            "Value (%)": round(adf, 2),
            "Status": adf_status
        },

        "IVDMD": {
            "Value (%)": round(ivdmd, 2),
            "Status": ivdmd_status
        },

        "Risk Flags": flags,

        "Recommendations": recommendations
    }

    return result


print("Feed Quality & Advisory Engine loaded successfully.")

In [ ]:
# STEP 43B — Test the advisory engine

test_result = assess_feed_quality(
    cp=8.5,
    ndf=62,
    adf=42,
    ivdmd=58
)

print("======================================")
print("FEED QUALITY ASSESSMENT")
print("======================================")

print("\nOverall Status:")
print(test_result["Overall Status"])

print("\nNutritional Indicators:")

for key in [
    "Crude Protein",
    "NDF",
    "ADF",
    "IVDMD"
]:
    print(
        f"{key}: "
        f"{test_result[key]['Value (%)']}% "
        f"→ {test_result[key]['Status']}"
    )

print("\nRisk Flags:")

if test_result["Risk Flags"]:
    for flag in test_result["Risk Flags"]:
        print("•", flag)
else:
    print("None")

print("\nRecommendations:")

if test_result["Recommendations"]:
    for recommendation in test_result["Recommendations"]:
        print("•", recommendation)
else:
    print("No immediate recommendation.")

In [ ]:
# STEP 44 — Find existing model prediction variables

print("======================================")
print("AVAILABLE PREDICTION VARIABLES")
print("======================================")

for name, value in globals().items():

    if (
        "pred" in name.lower()
        and hasattr(value, "shape")
    ):
        print(
            f"{name:35s} "
            f"shape={value.shape}"
        )

In [ ]:
# STEP 44B — Identify CP prediction variable

print("======================================")
print("POSSIBLE CP PREDICTIONS")
print("======================================")

for name in [
    "y_pred",
    "y_pred_final",
    "y_pred_ndf",
    "y_pred_adf",
    "y_pred_ivdmd"
]:

    if name in globals():
        value = globals()[name]

        print(f"\n{name}")
        print("Shape:", value.shape)
        print("First 10 predictions:")
        print(np.round(value[:10], 3))
        print("Min:", np.min(value))
        print("Max:", np.max(value))

In [ ]:
# STEP 44C — Find trained PLS models

from sklearn.cross_decomposition import PLSRegression

print("======================================")
print("TRAINED PLS MODELS")
print("======================================")

for name, value in globals().items():

    if isinstance(value, PLSRegression):

        print(f"\nModel variable: {name}")
        print(f"Number of components: {value.n_components}")

        if hasattr(value, "x_weights_"):
            print(
                f"Number of wavelengths/features: "
                f"{value.x_weights_.shape[0]}"
            )

In [ ]:
# STEP 44D — Complete NIR → Nutrition → Advisory Pipeline

import numpy as np

def predict_feed_quality_from_nir(nir_spectrum):
    """
    Complete AI pipeline:
    NIR spectrum
        ↓
    CP + NDF + ADF + IVDMD predictions
        ↓
    Feed quality advisory
    """

    # Convert input to numpy array
    nir_spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    # Check number of wavelengths
    if nir_spectrum.shape[1] != 1050:
        raise ValueError(
            f"Expected 1050 wavelength values, "
            f"but received {nir_spectrum.shape[1]}"
        )

    # -----------------------------------------
    # 1. NUTRITIONAL PREDICTIONS
    # -----------------------------------------

    cp_pred = float(
        final_cp_model.predict(nir_spectrum).ravel()[0]
    )

    ndf_pred = float(
        final_ndf_model.predict(nir_spectrum).ravel()[0]
    )

    adf_pred = float(
        final_adf_model.predict(nir_spectrum).ravel()[0]
    )

    ivdmd_pred = float(
        final_ivdmd_model.predict(nir_spectrum).ravel()[0]
    )

    # -----------------------------------------
    # 2. ADVISORY ENGINE
    # -----------------------------------------

    advisory = assess_feed_quality(
        cp=cp_pred,
        ndf=ndf_pred,
        adf=adf_pred,
        ivdmd=ivdmd_pred
    )

    # -----------------------------------------
    # 3. RETURN COMPLETE RESULT
    # -----------------------------------------

    return {
        "predictions": {
            "CP (%)": cp_pred,
            "NDF (%)": ndf_pred,
            "ADF (%)": adf_pred,
            "IVDMD (%)": ivdmd_pred
        },

        "advisory": advisory
    }


print("Complete NIR → AI → Advisory pipeline loaded.")

In [ ]:
# STEP 44E — Test complete pipeline on one real NIR sample

# Use the first test-set NIR spectrum
sample_spectrum = X_ivdmd_test[0]

result = predict_feed_quality_from_nir(
    sample_spectrum
)

print("==============================================")
print("COMPLETE FEED AI ASSESSMENT")
print("==============================================")

print("\nNUTRITIONAL PREDICTIONS")
print("----------------------------------------------")

for parameter, value in result["predictions"].items():
    print(f"{parameter}: {value:.2f}")

print("\nOVERALL QUALITY")
print("----------------------------------------------")

print(
    result["advisory"]["Overall Status"]
)

print("\nINDIVIDUAL STATUS")
print("----------------------------------------------")

for key in [
    "Crude Protein",
    "NDF",
    "ADF",
    "IVDMD"
]:

    item = result["advisory"][key]

    print(
        f"{key}: "
        f"{item['Value (%)']:.2f}% "
        f"→ {item['Status']}"
    )

print("\nRISK FLAGS")
print("----------------------------------------------")

if result["advisory"]["Risk Flags"]:

    for flag in result["advisory"]["Risk Flags"]:
        print("•", flag)

else:
    print("None")

print("\nRECOMMENDATIONS")
print("----------------------------------------------")

if result["advisory"]["Recommendations"]:

    for recommendation in result["advisory"]["Recommendations"]:
        print("•", recommendation)

else:
    print("No immediate recommendation.")

In [ ]:
# STEP 45 — Batch test the complete AI pipeline

print("==============================================")
print("BATCH AI PIPELINE TEST")
print("==============================================")

# Use the IVDMD test spectra
X_batch = X_ivdmd_test

results = []

for i in range(len(X_batch)):

    result = predict_feed_quality_from_nir(X_batch[i])

    results.append({
        "Sample": i + 1,
        "CP (%)": result["predictions"]["CP (%)"],
        "NDF (%)": result["predictions"]["NDF (%)"],
        "ADF (%)": result["predictions"]["ADF (%)"],
        "IVDMD (%)": result["predictions"]["IVDMD (%)"],
        "Overall Status": result["advisory"]["Overall Status"]
    })

batch_results = pd.DataFrame(results)

print("\nNumber of samples processed:", len(batch_results))

print("\nFirst 10 predictions:")
display(batch_results.head(10))

print("\nOverall status distribution:")
print(
    batch_results["Overall Status"]
    .value_counts()
)

In [ ]:
# STEP 46 — Final AI Prediction Interface
# Corrected for 1D PLS prediction output

def final_ai_prediction(nir_spectrum):

    # Convert input to numpy array
    nir_spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    # Check input
    if nir_spectrum.shape[1] != 1050:
        raise ValueError(
            f"Expected 1050 wavelength values, "
            f"received {nir_spectrum.shape[1]}"
        )

    # ---------------------------------
    # 1. ML PREDICTIONS
    # ---------------------------------

    cp = float(
        final_cp_model.predict(nir_spectrum).ravel()[0]
    )

    ndf = float(
        final_ndf_model.predict(nir_spectrum).ravel()[0]
    )

    adf = float(
        final_adf_model.predict(nir_spectrum).ravel()[0]
    )

    ivdmd = float(
        final_ivdmd_model.predict(nir_spectrum).ravel()[0]
    )

    # ---------------------------------
    # 2. ADVISORY ENGINE
    # ---------------------------------

    advisory = assess_feed_quality(
        cp=cp,
        ndf=ndf,
        adf=adf,
        ivdmd=ivdmd
    )

    # ---------------------------------
    # 3. FINAL RESULT
    # ---------------------------------

    return {
        "nutritional_predictions": {
            "crude_protein_percent": round(cp, 2),
            "ndf_percent": round(ndf, 2),
            "adf_percent": round(adf, 2),
            "ivdmd_percent": round(ivdmd, 2)
        },

        "quality_status": advisory["Overall Status"],

        "individual_status": {
            "crude_protein": advisory["Crude Protein"]["Status"],
            "ndf": advisory["NDF"]["Status"],
            "adf": advisory["ADF"]["Status"],
            "ivdmd": advisory["IVDMD"]["Status"]
        },

        "risk_flags": advisory["Risk Flags"],

        "recommendations": advisory["Recommendations"]
    }


print("Final AI prediction interface created successfully.")

In [ ]:
# STEP 46B — Test final AI interface

final_result = final_ai_prediction(X_ivdmd_test[0])

print("======================================")
print("FINAL AI RESULT")
print("======================================")

print("\nNutritional Predictions:")
for key, value in final_result["nutritional_predictions"].items():
    print(f"{key}: {value}")

print("\nQuality Status:")
print(final_result["quality_status"])

print("\nIndividual Status:")
for key, value in final_result["individual_status"].items():
    print(f"{key}: {value}")

print("\nRisk Flags:")
for item in final_result["risk_flags"]:
    print("•", item)

print("\nRecommendations:")
for item in final_result["recommendations"]:
    print("•", item)

In [ ]:
# STEP 47 — Batch test final AI interface

print("======================================")
print("BATCH TEST — FINAL AI INTERFACE")
print("======================================")

batch_results = []

for i in range(len(X_ivdmd_test)):

    result = final_ai_prediction(X_ivdmd_test[i])

    row = {
        "Sample": i + 1,

        "CP (%)": result["nutritional_predictions"]["crude_protein_percent"],
        "NDF (%)": result["nutritional_predictions"]["ndf_percent"],
        "ADF (%)": result["nutritional_predictions"]["adf_percent"],
        "IVDMD (%)": result["nutritional_predictions"]["ivdmd_percent"],

        "Quality": result["quality_status"]
    }

    batch_results.append(row)


batch_results = pd.DataFrame(batch_results)

print("\nSamples processed:", len(batch_results))

print("\nFirst 10 results:")
display(batch_results.head(10))

print("\nQuality distribution:")
print(
    batch_results["Quality"].value_counts()
)

In [ ]:
# STEP 48 — NIR Sample Similarity / Confidence Indicator

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("======================================")
print("BUILDING NIR CONFIDENCE SYSTEM")
print("======================================")

# Use training NIR data
X_train_array = np.asarray(X_ivdmd_train)

print("Training spectra:", X_train_array.shape)

# Standardize wavelengths
confidence_scaler = StandardScaler()

X_train_scaled = confidence_scaler.fit_transform(
    X_train_array
)

# Reduce spectral space
confidence_pca = PCA(
    n_components=10,
    random_state=42
)

X_train_pca = confidence_pca.fit_transform(
    X_train_scaled
)

print(
    "PCA variance retained:",
    round(
        confidence_pca.explained_variance_ratio_.sum() * 100,
        2
    ),
    "%"
)

# Calculate center of training distribution
training_center = np.mean(
    X_train_pca,
    axis=0
)

# Distance of every training sample from center
training_distances = np.linalg.norm(
    X_train_pca - training_center,
    axis=1
)

# Reference distance threshold
confidence_threshold = np.percentile(
    training_distances,
    95
)

print(
    "95th percentile distance:",
    round(confidence_threshold, 3)
)

print("\nConfidence system ready.")

In [ ]:
# STEP 48B — Add spectral confidence to final AI prediction

def calculate_spectral_confidence(nir_spectrum):

    # Convert to correct shape
    spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    # Transform using training-data scaler
    spectrum_scaled = confidence_scaler.transform(
        spectrum
    )

    # Project into PCA space
    spectrum_pca = confidence_pca.transform(
        spectrum_scaled
    )

    # Distance from training distribution center
    distance = np.linalg.norm(
        spectrum_pca[0] - training_center
    )

    # Convert distance into a simple confidence indicator
    if distance <= confidence_threshold:
        confidence_status = "HIGH"
    else:
        confidence_status = "LOW"

    return {
        "spectral_distance": round(float(distance), 3),
        "confidence_status": confidence_status
    }


# -----------------------------------------
# TEST CONFIDENCE SYSTEM
# -----------------------------------------

confidence_test = calculate_spectral_confidence(
    X_ivdmd_test[0]
)

print("======================================")
print("SPECTRAL CONFIDENCE TEST")
print("======================================")

print(
    "Spectral distance:",
    confidence_test["spectral_distance"]
)

print(
    "95% training threshold:",
    round(confidence_threshold, 3)
)

print(
    "Confidence status:",
    confidence_test["confidence_status"]
)

In [ ]:
# STEP 48C — Final AI Prediction + Spectral Confidence

def final_ai_prediction(nir_spectrum):

    # ---------------------------------
    # 1. INPUT CHECK
    # ---------------------------------

    nir_spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    if nir_spectrum.shape[1] != 1050:
        raise ValueError(
            f"Expected 1050 wavelength values, "
            f"received {nir_spectrum.shape[1]}"
        )

    # ---------------------------------
    # 2. NUTRITIONAL PREDICTIONS
    # ---------------------------------

    cp = float(
        final_cp_model.predict(nir_spectrum).ravel()[0]
    )

    ndf = float(
        final_ndf_model.predict(nir_spectrum).ravel()[0]
    )

    adf = float(
        final_adf_model.predict(nir_spectrum).ravel()[0]
    )

    ivdmd = float(
        final_ivdmd_model.predict(nir_spectrum).ravel()[0]
    )

    # ---------------------------------
    # 3. SPECTRAL CONFIDENCE
    # ---------------------------------

    confidence = calculate_spectral_confidence(
        nir_spectrum
    )

    # ---------------------------------
    # 4. ADVISORY ENGINE
    # ---------------------------------

    advisory = assess_feed_quality(
        cp=cp,
        ndf=ndf,
        adf=adf,
        ivdmd=ivdmd
    )

    # ---------------------------------
    # 5. FINAL RESULT
    # ---------------------------------

    return {

        "nutritional_predictions": {
            "crude_protein_percent": round(cp, 2),
            "ndf_percent": round(ndf, 2),
            "adf_percent": round(adf, 2),
            "ivdmd_percent": round(ivdmd, 2)
        },

        "spectral_confidence": {
            "status": confidence["confidence_status"],
            "distance": confidence["spectral_distance"],
            "threshold": round(
                float(confidence_threshold), 3
            )
        },

        "quality_status": advisory["Overall Status"],

        "individual_status": {
            "crude_protein": advisory["Crude Protein"]["Status"],
            "ndf": advisory["NDF"]["Status"],
            "adf": advisory["ADF"]["Status"],
            "ivdmd": advisory["IVDMD"]["Status"]
        },

        "risk_flags": advisory["Risk Flags"],

        "recommendations": advisory["Recommendations"]
    }


print("Final AI system updated with spectral confidence.")

In [ ]:
# STEP 48D — Test final AI system

final_result = final_ai_prediction(
    X_ivdmd_test[0]
)

print("======================================")
print("FINAL AI SYSTEM RESULT")
print("======================================")

print("\nNUTRITIONAL PREDICTIONS")
print("--------------------------------------")

for key, value in final_result[
    "nutritional_predictions"
].items():

    print(f"{key}: {value}")

print("\nSPECTRAL CONFIDENCE")
print("--------------------------------------")

print(
    "Status:",
    final_result["spectral_confidence"]["status"]
)

print(
    "Distance:",
    final_result["spectral_confidence"]["distance"]
)

print(
    "Threshold:",
    final_result["spectral_confidence"]["threshold"]
)

print("\nQUALITY STATUS")
print("--------------------------------------")

print(
    final_result["quality_status"]
)

print("\nINDIVIDUAL STATUS")
print("--------------------------------------")

for key, value in final_result[
    "individual_status"
].items():

    print(f"{key}: {value}")

print("\nRISK FLAGS")
print("--------------------------------------")

if final_result["risk_flags"]:

    for item in final_result["risk_flags"]:
        print("•", item)

else:

    print("None")

print("\nRECOMMENDATIONS")
print("--------------------------------------")

if final_result["recommendations"]:

    for item in final_result["recommendations"]:
        print("•", item)

else:

    print("No immediate recommendation.")

In [ ]:
# STEP 49 — SAVE FINAL AI MODELS

import joblib
import os

print("======================================")
print("SAVING FINAL AI MODELS")
print("======================================")

model_dir = "/content/feed_ai_models"
os.makedirs(model_dir, exist_ok=True)

# Save nutritional models
joblib.dump(
    final_cp_model,
    f"{model_dir}/cp_pls_model.pkl"
)

joblib.dump(
    final_ndf_model,
    f"{model_dir}/ndf_pls_model.pkl"
)

joblib.dump(
    final_adf_model,
    f"{model_dir}/adf_pls_model.pkl"
)

joblib.dump(
    final_ivdmd_model,
    f"{model_dir}/ivdmd_pls_model.pkl"
)

# Save confidence system
joblib.dump(
    confidence_scaler,
    f"{model_dir}/confidence_scaler.pkl"
)

joblib.dump(
    confidence_pca,
    f"{model_dir}/confidence_pca.pkl"
)

# Save confidence threshold
joblib.dump(
    confidence_threshold,
    f"{model_dir}/confidence_threshold.pkl"
)

# Save wavelength information
joblib.dump(
    wavelength_cols,
    f"{model_dir}/wavelength_cols.pkl"
)

print("\nSaved files:")

for file in sorted(os.listdir(model_dir)):
    print("•", file)

print("\nModel directory:")
print(model_dir)

In [ ]:
# STEP 50 — Create reusable AI predictor

def feed_ai_predict(nir_spectrum):

    # -------------------------------
    # 1. Prepare input
    # -------------------------------

    spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    if spectrum.shape[1] != 1050:
        raise ValueError(
            f"Expected 1050 wavelength values, "
            f"received {spectrum.shape[1]}"
        )

    # -------------------------------
    # 2. Nutritional predictions
    # -------------------------------

    cp = float(
        final_cp_model.predict(spectrum).ravel()[0]
    )

    ndf = float(
        final_ndf_model.predict(spectrum).ravel()[0]
    )

    adf = float(
        final_adf_model.predict(spectrum).ravel()[0]
    )

    ivdmd = float(
        final_ivdmd_model.predict(spectrum).ravel()[0]
    )

    # -------------------------------
    # 3. Spectral confidence
    # -------------------------------

    confidence = calculate_spectral_confidence(
        spectrum
    )

    # -------------------------------
    # 4. Quality assessment
    # -------------------------------

    advisory = assess_feed_quality(
        cp=cp,
        ndf=ndf,
        adf=adf,
        ivdmd=ivdmd
    )

    # -------------------------------
    # 5. Return unified result
    # -------------------------------

    return {
        "predictions": {
            "cp_percent": round(cp, 2),
            "ndf_percent": round(ndf, 2),
            "adf_percent": round(adf, 2),
            "ivdmd_percent": round(ivdmd, 2)
        },

        "spectral_confidence": {
            "status": confidence["confidence_status"],
            "distance": confidence["spectral_distance"],
            "threshold": round(
                float(confidence_threshold), 3
            )
        },

        "quality_status": advisory["Overall Status"],

        "individual_status": {
            "crude_protein":
                advisory["Crude Protein"]["Status"],

            "ndf":
                advisory["NDF"]["Status"],

            "adf":
                advisory["ADF"]["Status"],

            "ivdmd":
                advisory["IVDMD"]["Status"]
        },

        "risk_flags":
            advisory["Risk Flags"],

        "recommendations":
            advisory["Recommendations"]
    }


print("======================================")
print("REUSABLE FEED AI PREDICTOR")
print("======================================")
print("AI predictor created successfully.")

In [ ]:
# STEP 50B — Test reusable predictor

test_prediction = feed_ai_predict(
    X_ivdmd_test[0]
)

print("\n======================================")
print("STEP 50 TEST RESULT")
print("======================================")

print("\nPredictions:")
print(test_prediction["predictions"])

print("\nSpectral confidence:")
print(test_prediction["spectral_confidence"])

print("\nQuality:")
print(test_prediction["quality_status"])

print("\nRisk flags:")
for flag in test_prediction["risk_flags"]:
    print("•", flag)

print("\nRecommendations:")
for recommendation in test_prediction["recommendations"]:
    print("•", recommendation)

In [ ]:
# STEP 51 — Create deployment package

import shutil
import os

print("======================================")
print("CREATING AI DEPLOYMENT PACKAGE")
print("======================================")

source_dir = "/content/feed_ai_models"
package_dir = "/content/feed_ai_deployment"

# Remove old package if it exists
if os.path.exists(package_dir):
    shutil.rmtree(package_dir)

# Create package directory
os.makedirs(package_dir)

# Copy trained model files
for filename in os.listdir(source_dir):

    source_path = os.path.join(
        source_dir,
        filename
    )

    destination_path = os.path.join(
        package_dir,
        filename
    )

    shutil.copy2(
        source_path,
        destination_path
    )

# Create README
readme_text = """
FEED QUALITY AI MODEL
=====================

Models:
- CP PLS regression
- NDF PLS regression
- ADF PLS regression
- IVDMD PLS regression

Confidence system:
- StandardScaler
- PCA
- Spectral distance threshold

Input:
- 1050 NIR wavelength values
- 400–2498 nm
- 2 nm spacing

Outputs:
- Crude Protein (%)
- NDF (%)
- ADF (%)
- IVDMD (%)
- Spectral confidence
- Quality status
- Risk flags
- Recommendations

Important:
These models were trained using a forage NIR dataset.
They should be locally validated against laboratory
reference measurements before real-world deployment.
"""

with open(
    os.path.join(package_dir, "README.txt"),
    "w"
) as f:
    f.write(readme_text)

print("\nDeployment package created:")
print(package_dir)

print("\nFiles:")

for filename in sorted(
    os.listdir(package_dir)
):
    print("•", filename)

In [ ]:
# STEP 52 — ZIP AI deployment package

import shutil
import os

print("======================================")
print("PACKAGING AI FOR DEPLOYMENT")
print("======================================")

zip_path = shutil.make_archive(
    "/content/feed_ai_deployment",
    "zip",
    "/content/feed_ai_deployment"
)

print("\nZIP created:")
print(zip_path)

print("\nPackage size:")
print(
    round(
        os.path.getsize(zip_path) / (1024 * 1024),
        2
    ),
    "MB"
)

In [ ]:
# STEP 52B — Download package

from google.colab import files

files.download(
    "/content/feed_ai_deployment.zip"
)

Send this API Package Link to Neha for the Backend Part

In [ ]:
# STEP 51.1 — Rebuild IVDMD dataset from the original FullData file

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the original FullData dataset
full_data_path = "/content/02a._Fulldata[1].tab"

df_ivdmd = pd.read_csv(full_data_path, sep="\t")

print("Full dataset shape:", df_ivdmd.shape)

# IVDMD target
target_col = "DIVMS"

# Identify the 1050 NIR wavelength columns
wavelength_cols_ivdmd = [
    col for col in df_ivdmd.columns
    if str(col).replace(".", "", 1).isdigit()
]

print("Number of wavelength columns found:", len(wavelength_cols_ivdmd))
print("Target column:", target_col)

# Keep only rows where IVDMD and all NIR values are available
ivdmd_data = df_ivdmd[wavelength_cols_ivdmd + [target_col]].dropna()

# Create X and y
X_ivdmd = ivdmd_data[wavelength_cols_ivdmd].values
y_ivdmd = ivdmd_data[target_col].values

print("\nIVDMD X shape:", X_ivdmd.shape)
print("IVDMD y shape:", y_ivdmd.shape)

print("\nIVDMD statistics:")
print("Mean:", np.mean(y_ivdmd))
print("Std:", np.std(y_ivdmd))
print("Min:", np.min(y_ivdmd))
print("Median:", np.median(y_ivdmd))
print("Max:", np.max(y_ivdmd))

# Recreate the same 80/20 split
X_ivdmd_train, X_ivdmd_test, y_ivdmd_train, y_ivdmd_test = train_test_split(
    X_ivdmd,
    y_ivdmd,
    test_size=0.20,
    random_state=42
)

print("\nTraining shape:", X_ivdmd_train.shape)
print("Test shape:", X_ivdmd_test.shape)

In [ ]:
import os

print("Files in /content:")
for f in os.listdir("/content"):
    print(f)

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import os

print("Files in /content:")
for f in os.listdir("/content"):
    print(repr(f))

In [ ]:
# STEP 51.1 — Rebuild IVDMD dataset

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load original FullData
full_data_path = "/content/02a._Fulldata[1].tab"

df_ivdmd = pd.read_csv(full_data_path, sep="\t")

print("Full dataset shape:", df_ivdmd.shape)

# IVDMD target
target_col = "DIVMS"

# Convert target column to numeric, coercing errors to NaN
df_ivdmd[target_col] = pd.to_numeric(df_ivdmd[target_col], errors='coerce')

# Detect the 1050 wavelength columns
wavelength_cols_ivdmd = [
    col for col in df_ivdmd.columns
    if str(col).replace(".", "", 1).isdigit()
]

print("Number of wavelength columns:", len(wavelength_cols_ivdmd))
print("Target column:", target_col)

# Keep rows with complete NIR + IVDMD data
ivdmd_data = df_ivdmd[wavelength_cols_ivdmd + [target_col]].dropna()

# Create X and y
X_ivdmd = ivdmd_data[wavelength_cols_ivdmd].values
y_ivdmd = ivdmd_data[target_col].values

print("\nIVDMD X shape:", X_ivdmd.shape)
print("IVDMD y shape:", y_ivdmd.shape)

print("\nIVDMD statistics:")
print("Mean:", np.mean(y_ivdmd))
print("Std:", np.std(y_ivdmd))
print("Min:", np.min(y_ivdmd))
print("Median:", np.median(y_ivdmd))
print("Max:", np.max(y_ivdmd))

# Same 80/20 split used previously
X_ivdmd_train, X_ivdmd_test, y_ivdmd_train, y_ivdmd_test = train_test_split(
    X_ivdmd,
    y_ivdmd,
    test_size=0.20,
    random_state=42
)

print("\nTraining shape:", X_ivdmd_train.shape)
print("Test shape:", X_ivdmd_test.shape)


In [ ]:
# STEP 51.2 — Standardize NIR spectra and apply PCA

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize using training data only
anomaly_scaler = StandardScaler()

X_ivdmd_train_scaled = anomaly_scaler.fit_transform(X_ivdmd_train)

# Transform test data using the same scaler
X_ivdmd_test_scaled = anomaly_scaler.transform(X_ivdmd_test)

print("Scaled training shape:", X_ivdmd_train_scaled.shape)
print("Scaled test shape:", X_ivdmd_test_scaled.shape)

# PCA: keep enough components to explain 95% of variance
anomaly_pca = PCA(n_components=0.95)

X_ivdmd_train_pca = anomaly_pca.fit_transform(X_ivdmd_train_scaled)
X_ivdmd_test_pca = anomaly_pca.transform(X_ivdmd_test_scaled)

print("\nPCA training shape:", X_ivdmd_train_pca.shape)
print("PCA test shape:", X_ivdmd_test_pca.shape)

print("\nNumber of PCA components:", anomaly_pca.n_components_)
print("Explained variance:", anomaly_pca.explained_variance_ratio_.sum())


In [ ]:
# STEP 51.2 — Standardize NIR spectra and apply PCA
# Corrected: explicitly define X_anomaly_train first

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Use the IVDMD training spectra for anomaly detection
X_anomaly_train = X_ivdmd_train.copy()

# Standardize training data
anomaly_scaler = StandardScaler()

X_anomaly_train_scaled = anomaly_scaler.fit_transform(
    X_anomaly_train
)

# Standardize test data using the SAME scaler
X_anomaly_test_scaled = anomaly_scaler.transform(
    X_ivdmd_test
)

print("X_anomaly_train shape:", X_anomaly_train.shape)
print("Scaled training shape:", X_anomaly_train_scaled.shape)
print("Scaled test shape:", X_anomaly_test_scaled.shape)

# PCA retaining 95% of spectral variance
anomaly_pca = PCA(n_components=0.95)

X_anomaly_train_pca = anomaly_pca.fit_transform(
    X_anomaly_train_scaled
)

X_anomaly_test_pca = anomaly_pca.transform(
    X_anomaly_test_scaled
)

print("\nPCA training shape:", X_anomaly_train_pca.shape)
print("PCA test shape:", X_anomaly_test_pca.shape)

print("\nNumber of PCA components:", anomaly_pca.n_components_)
print(
    "Explained variance:",
    anomaly_pca.explained_variance_ratio_.sum()
)

In [ ]:
# STEP 51.3 — Train Isolation Forest anomaly detector

from sklearn.ensemble import IsolationForest

# Create the anomaly detector
anomaly_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

# Train ONLY on the normal/reference training distribution
anomaly_model.fit(X_anomaly_train_pca)

print("Isolation Forest trained successfully.")
print("Number of trees:", anomaly_model.n_estimators)
print("Contamination setting:", anomaly_model.contamination)

In [ ]:
# STEP 51.4 — Detect anomalies in training and test spectra

# Predict anomaly labels
train_anomaly_labels = anomaly_model.predict(X_anomaly_train_pca)
test_anomaly_labels = anomaly_model.predict(X_anomaly_test_pca)

# Convert:
#  1  -> NORMAL
# -1  -> ANOMALOUS

train_status = np.where(
    train_anomaly_labels == 1,
    "NORMAL",
    "ANOMALOUS"
)

test_status = np.where(
    test_anomaly_labels == 1,
    "NORMAL",
    "ANOMALOUS"
)

# Count results
train_normal = np.sum(train_anomaly_labels == 1)
train_anomalous = np.sum(train_anomaly_labels == -1)

test_normal = np.sum(test_anomaly_labels == 1)
test_anomalous = np.sum(test_anomaly_labels == -1)

print("TRAINING DATA")
print("----------------")
print("Normal samples:", train_normal)
print("Anomalous samples:", train_anomalous)

print("\nTEST DATA")
print("----------------")
print("Normal samples:", test_normal)
print("Anomalous samples:", test_anomalous)

In [ ]:
# STEP 51.5 — Calculate anomaly scores

# Isolation Forest decision scores
train_anomaly_scores = anomaly_model.decision_function(
    X_anomaly_train_pca
)

test_anomaly_scores = anomaly_model.decision_function(
    X_anomaly_test_pca
)

# Lower score = more anomalous
print("TRAINING ANOMALY SCORES")
print("------------------------")
print("Minimum:", train_anomaly_scores.min())
print("Maximum:", train_anomaly_scores.max())
print("Mean:", train_anomaly_scores.mean())

print("\nTEST ANOMALY SCORES")
print("--------------------")
print("Minimum:", test_anomaly_scores.min())
print("Maximum:", test_anomaly_scores.max())
print("Mean:", test_anomaly_scores.mean())

# Show the 10 most unusual test samples
most_anomalous_indices = np.argsort(test_anomaly_scores)[:10]

print("\n10 MOST ANOMALOUS TEST SAMPLES")
print("--------------------------------")
for rank, idx in enumerate(most_anomalous_indices, start=1):
    print(
        f"{rank}. Test sample index: {idx}, "
        f"Score: {test_anomaly_scores[idx]:.4f}"
    )

In [ ]:
import os

print("Files currently in /content:")
for f in os.listdir("/content"):
    print(f)

In [ ]:
# STEP 51.6 — Restore the previously trained nutritional models

import os
import zipfile
import joblib

# Deployment ZIP
zip_path = "/content/feed_ai_deployment.zip"

# Extract deployment package
deployment_dir = "/content/feed_ai_deployment"

if not os.path.exists(deployment_dir):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(deployment_dir)

print("Deployment files:")
for f in os.listdir(deployment_dir):
    print(" -", f)

# Load the four trained PLS models
final_cp_model = joblib.load(
    os.path.join(deployment_dir, "cp_pls_model.pkl")
)

final_ndf_model = joblib.load(
    os.path.join(deployment_dir, "ndf_pls_model.pkl")
)

final_adf_model = joblib.load(
    os.path.join(deployment_dir, "adf_pls_model.pkl")
)

final_ivdmd_model = joblib.load(
    os.path.join(deployment_dir, "ivdmd_pls_model.pkl")
)

print("\nAll four nutritional models loaded successfully!")

print("CP model components:", final_cp_model.n_components)
print("NDF model components:", final_ndf_model.n_components)
print("ADF model components:", final_adf_model.n_components)
print("IVDMD model components:", final_ivdmd_model.n_components)

In [ ]:
# STEP 51.7 — Compare nutritional predictions
# between normal and anomalous NIR spectra

# Predict all four nutritional parameters
cp_test_pred = final_cp_model.predict(X_ivdmd_test).ravel()
ndf_test_pred = final_ndf_model.predict(X_ivdmd_test).ravel()
adf_test_pred = final_adf_model.predict(X_ivdmd_test).ravel()
ivdmd_test_pred = final_ivdmd_model.predict(X_ivdmd_test).ravel()

# Masks from our Isolation Forest
normal_mask = test_anomaly_labels == 1
anomaly_mask = test_anomaly_labels == -1

print("NORMAL SAMPLES")
print("================")
print("Count:", normal_mask.sum())
print("Mean CP:", round(cp_test_pred[normal_mask].mean(), 3))
print("Mean NDF:", round(ndf_test_pred[normal_mask].mean(), 3))
print("Mean ADF:", round(adf_test_pred[normal_mask].mean(), 3))
print("Mean IVDMD:", round(ivdmd_test_pred[normal_mask].mean(), 3))

print("\nANOMALOUS SAMPLES")
print("===================")
print("Count:", anomaly_mask.sum())
print("Mean CP:", round(cp_test_pred[anomaly_mask].mean(), 3))
print("Mean NDF:", round(ndf_test_pred[anomaly_mask].mean(), 3))
print("Mean ADF:", round(adf_test_pred[anomaly_mask].mean(), 3))
print("Mean IVDMD:", round(ivdmd_test_pred[anomaly_mask].mean(), 3))

In [ ]:
# STEP 51.8 — Compare IVDMD prediction error
# between normal and anomalous samples

# Actual laboratory IVDMD values
ivdmd_actual = y_ivdmd_test

# Absolute prediction error
ivdmd_error = np.abs(ivdmd_actual - ivdmd_test_pred)

# Calculate error for each group
normal_error = ivdmd_error[normal_mask]
anomaly_error = ivdmd_error[anomaly_mask]

print("NORMAL SAMPLES")
print("================")
print("Count:", len(normal_error))
print("Mean absolute error:", round(normal_error.mean(), 3))
print("Median absolute error:", round(np.median(normal_error), 3))
print("Maximum error:", round(normal_error.max(), 3))

print("\nANOMALOUS SAMPLES")
print("===================")
print("Count:", len(anomaly_error))
print("Mean absolute error:", round(anomaly_error.mean(), 3))
print("Median absolute error:", round(np.median(anomaly_error), 3))
print("Maximum error:", round(anomaly_error.max(), 3))

In [ ]:
# STEP 51.9 — Inspect the largest IVDMD prediction errors

# Find the 10 samples with the largest prediction errors
worst_error_indices = np.argsort(ivdmd_error)[-10:][::-1]

print("10 LARGEST IVDMD PREDICTION ERRORS")
print("====================================")

for rank, idx in enumerate(worst_error_indices, start=1):
    status = "ANOMALOUS" if test_anomaly_labels[idx] == -1 else "NORMAL"

    print(
        f"{rank}. Test index: {idx} | "
        f"Actual: {ivdmd_actual[idx]:.2f} | "
        f"Predicted: {ivdmd_test_pred[idx]:.2f} | "
        f"Error: {ivdmd_error[idx]:.2f} | "
        f"Status: {status} | "
        f"Anomaly score: {test_anomaly_scores[idx]:.4f}"
    )

In [ ]:
# STEP 51.10 — Robust error diagnostic
# Examine the effect of the extreme 108.53 IVDMD observation

# Identify the extreme IVDMD observation
extreme_mask = ivdmd_actual > 100

print("Extreme IVDMD samples (>100):", extreme_mask.sum())

if extreme_mask.sum() > 0:
    print("\nExtreme sample details:")
    for idx in np.where(extreme_mask)[0]:
        print(
            f"Test index: {idx} | "
            f"Actual: {ivdmd_actual[idx]:.2f} | "
            f"Predicted: {ivdmd_test_pred[idx]:.2f} | "
            f"Error: {ivdmd_error[idx]:.2f} | "
            f"Anomaly status: "
            f"{'ANOMALOUS' if test_anomaly_labels[idx] == -1 else 'NORMAL'}"
        )

# Diagnostic only — do NOT use this as the official model score
diagnostic_mask = ~extreme_mask

print("\nOFFICIAL TEST MAE:")
print(round(ivdmd_error.mean(), 3))

print("\nDIAGNOSTIC MAE WITHOUT >100 IVDMD:")
print(round(ivdmd_error[diagnostic_mask].mean(), 3))

print("\nOfficial test sample count:", len(ivdmd_error))
print("Diagnostic sample count:", diagnostic_mask.sum())

In [ ]:
# STEP 51.11 — Calculate PCA distance from the training distribution

import numpy as np

# Calculate the center of the training data in PCA space
pca_training_center = X_anomaly_train_pca.mean(axis=0)

# Calculate Euclidean distance from the training center
train_pca_distance = np.linalg.norm(
    X_anomaly_train_pca - pca_training_center,
    axis=1
)

test_pca_distance = np.linalg.norm(
    X_anomaly_test_pca - pca_training_center,
    axis=1
)

# Set a threshold using the 95th percentile of training distances
pca_distance_threshold = np.percentile(
    train_pca_distance,
    95
)

print("PCA distance threshold:", round(pca_distance_threshold, 3))

print("\nTRAINING DISTANCE")
print("------------------")
print("Minimum:", round(train_pca_distance.min(), 3))
print("Maximum:", round(train_pca_distance.max(), 3))
print("Mean:", round(train_pca_distance.mean(), 3))

print("\nTEST DISTANCE")
print("----------------")
print("Minimum:", round(test_pca_distance.min(), 3))
print("Maximum:", round(test_pca_distance.max(), 3))
print("Mean:", round(test_pca_distance.mean(), 3))

In [ ]:
# STEP 51.12 — Combine Isolation Forest + PCA distance

# PCA-distance flag
pca_anomaly_flag = test_pca_distance > pca_distance_threshold

# Isolation Forest flag
isolation_anomaly_flag = test_anomaly_labels == -1

# Combined warning:
# A sample is strongly unusual if BOTH methods agree.
combined_anomaly_flag = (
    isolation_anomaly_flag &
    pca_anomaly_flag
)

print("ANOMALY DETECTION COMPARISON")
print("============================")

print("Isolation Forest anomalies:",
      isolation_anomaly_flag.sum())

print("PCA distance anomalies:",
      pca_anomaly_flag.sum())

print("Both methods agree:",
      combined_anomaly_flag.sum())

print("\nAgreement rate among Isolation Forest anomalies:",
      round(
          combined_anomaly_flag[isolation_anomaly_flag].mean() * 100,
          2
      ),
      "%"
      )

In [ ]:
# STEP 51.13 — Inspect samples flagged by BOTH methods

consensus_indices = np.where(combined_anomaly_flag)[0]

print("CONSENSUS ANOMALIES")
print("===================")
print("Number of samples:", len(consensus_indices))

for rank, idx in enumerate(consensus_indices, start=1):

    print(f"\n--- Sample {rank} | Test index: {idx} ---")

    print(
        "Isolation Forest score:",
        round(test_anomaly_scores[idx], 4)
    )

    print(
        "PCA distance:",
        round(test_pca_distance[idx], 3)
    )

    print(
        "CP prediction:",
        round(cp_test_pred[idx], 2)
    )

    print(
        "NDF prediction:",
        round(ndf_test_pred[idx], 2)
    )

    print(
        "ADF prediction:",
        round(adf_test_pred[idx], 2)
    )

    print(
        "IVDMD actual:",
        round(ivdmd_actual[idx], 2)
    )

    print(
        "IVDMD predicted:",
        round(ivdmd_test_pred[idx], 2)
    )

    print(
        "IVDMD absolute error:",
        round(ivdmd_error[idx], 2)
    )

In [ ]:
# STEP 51.14 — Final reusable NIR anomaly detector

def nir_anomaly_detector(nir_spectrum):
    """
    Detect whether a NIR spectrum is unusual
    compared with the training spectral distribution.

    Input:
        nir_spectrum -> 1050 NIR wavelength values

    Output:
        Dictionary containing anomaly results.
    """

    # Convert input to numpy array
    spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    # Check number of wavelengths
    if spectrum.shape[1] != X_anomaly_train.shape[1]:
        raise ValueError(
            f"Expected {X_anomaly_train.shape[1]} NIR values, "
            f"but received {spectrum.shape[1]}"
        )

    # Standardize using training scaler
    spectrum_scaled = anomaly_scaler.transform(spectrum)

    # Project into PCA space
    spectrum_pca = anomaly_pca.transform(spectrum_scaled)

    # Isolation Forest prediction
    isolation_prediction = anomaly_model.predict(
        spectrum_pca
    )[0]

    isolation_score = anomaly_model.decision_function(
        spectrum_pca
    )[0]

    # PCA distance from training center
    pca_distance = np.linalg.norm(
        spectrum_pca[0] - pca_training_center
    )

    # Individual anomaly flags
    isolation_anomaly = isolation_prediction == -1
    pca_anomaly = pca_distance > pca_distance_threshold

    # Strong warning only when BOTH methods agree
    combined_anomaly = (
        isolation_anomaly and pca_anomaly
    )

    return {
        "isolation_forest_status":
            "ANOMALOUS" if isolation_anomaly else "NORMAL",

        "isolation_forest_score":
            float(isolation_score),

        "pca_distance":
            float(pca_distance),

        "pca_distance_status":
            "ANOMALOUS" if pca_anomaly else "NORMAL",

        "combined_status":
            "STRONG_SPECTRAL_WARNING"
            if combined_anomaly
            else "NO_STRONG_SPECTRAL_WARNING"
    }


print("NIR anomaly detector created successfully.")

In [ ]:
# STEP 51.15 — Test anomaly detector on a real test sample

sample_index = 9

sample_spectrum = X_ivdmd_test[sample_index]

anomaly_result = nir_anomaly_detector(sample_spectrum)

print("NIR ANOMALY TEST")
print("================")

print("Test sample index:", sample_index)

for key, value in anomaly_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# STEP 51.16 — Save NIR anomaly detection package

import os
import joblib

anomaly_package_dir = "/content/nir_anomaly_detection"

os.makedirs(anomaly_package_dir, exist_ok=True)

# Save the preprocessing objects
joblib.dump(
    anomaly_scaler,
    os.path.join(anomaly_package_dir, "anomaly_scaler.pkl")
)

joblib.dump(
    anomaly_pca,
    os.path.join(anomaly_package_dir, "anomaly_pca.pkl")
)

# Save Isolation Forest
joblib.dump(
    anomaly_model,
    os.path.join(anomaly_package_dir, "isolation_forest.pkl")
)

# Save PCA training center
joblib.dump(
    pca_training_center,
    os.path.join(anomaly_package_dir, "pca_training_center.pkl")
)

# Save PCA distance threshold
joblib.dump(
    pca_distance_threshold,
    os.path.join(anomaly_package_dir, "pca_distance_threshold.pkl")
)

print("NIR anomaly detection package saved successfully!")

print("\nSaved files:")
for f in os.listdir(anomaly_package_dir):
    print(" -", f)

In [ ]:
# STEP 51.17 — Combined Nutritional + Anomaly AI

def feed_ai_predict_v2(nir_spectrum):
    """
    Combined AI prediction for one NIR spectrum.

    Returns:
        - CP
        - NDF
        - ADF
        - IVDMD
        - Spectral anomaly information
        - Quality/risk information
    """

    spectrum = np.asarray(nir_spectrum).reshape(1, -1)

    # --------------------------------------------------
    # 1. Validate input
    # --------------------------------------------------
    expected_features = X_anomaly_train.shape[1]

    if spectrum.shape[1] != expected_features:
        raise ValueError(
            f"Expected {expected_features} NIR values, "
            f"but received {spectrum.shape[1]}"
        )

    # --------------------------------------------------
    # 2. Nutritional predictions
    # --------------------------------------------------
    cp = float(final_cp_model.predict(spectrum).ravel()[0])
    ndf = float(final_ndf_model.predict(spectrum).ravel()[0])
    adf = float(final_adf_model.predict(spectrum).ravel()[0])
    ivdmd = float(final_ivdmd_model.predict(spectrum).ravel()[0])

    # --------------------------------------------------
    # 3. Anomaly detection
    # --------------------------------------------------
    anomaly_result = nir_anomaly_detector(nir_spectrum)

    # --------------------------------------------------
    # 4. Existing quality assessment
    # --------------------------------------------------
    quality_result = assess_feed_quality(
        cp,
        ndf,
        adf,
        ivdmd
    )

    # --------------------------------------------------
    # 5. Final combined result
    # --------------------------------------------------
    return {
        "nutritional_predictions": {
            "CP": round(cp, 3),
            "NDF": round(ndf, 3),
            "ADF": round(adf, 3),
            "IVDMD": round(ivdmd, 3)
        },

        "spectral_anomaly": anomaly_result,

        "quality_assessment": quality_result
    }


print("Combined Nutritional + Anomaly AI created successfully.")

In [ ]:
# STEP 51.18 FIX — Restore feed quality assessment function

def assess_feed_quality(cp, ndf, adf, ivdmd):

    # Individual nutritional status
    if cp < 7:
        cp_status = "LOW"
    elif cp <= 12:
        cp_status = "ADEQUATE"
    else:
        cp_status = "HIGH"

    if ndf > 60:
        ndf_status = "HIGH"
    elif ndf >= 40:
        ndf_status = "MODERATE"
    else:
        ndf_status = "LOW"

    if adf > 40:
        adf_status = "HIGH"
    elif adf >= 30:
        adf_status = "MODERATE"
    else:
        adf_status = "LOW"

    if ivdmd < 55:
        ivdmd_status = "LOW"
    elif ivdmd <= 65:
        ivdmd_status = "MODERATE"
    else:
        ivdmd_status = "GOOD"

    # Risk flags
    risk_flags = []

    if cp_status == "LOW":
        risk_flags.append("Low crude protein")

    if ndf_status == "HIGH":
        risk_flags.append("High fiber (NDF)")

    if adf_status == "HIGH":
        risk_flags.append("High fiber (ADF)")

    if ivdmd_status == "LOW":
        risk_flags.append("Low digestibility")

    # Overall status
    if len(risk_flags) >= 2:
        quality_status = "NEEDS ATTENTION"
    elif len(risk_flags) == 1:
        quality_status = "MODERATE"
    else:
        quality_status = "GOOD"

    # Recommendations
    recommendations = []

    if cp_status == "LOW":
        recommendations.append(
            "Consider protein supplementation."
        )

    if ndf_status == "HIGH":
        recommendations.append(
            "Review forage maturity, intake and fiber level."
        )

    if adf_status == "HIGH":
        recommendations.append(
            "High ADF may reduce feed digestibility."
        )

    if ivdmd_status == "LOW":
        recommendations.append(
            "Low digestibility detected; review forage quality."
        )

    if not recommendations:
        recommendations.append(
            "Nutritional profile appears acceptable."
        )

    return {
        "quality_status": quality_status,

        "individual_status": {
            "CP": cp_status,
            "NDF": ndf_status,
            "ADF": adf_status,
            "IVDMD": ivdmd_status
        },

        "risk_flags": risk_flags,

        "recommendations": recommendations
    }


print("Quality assessment function restored successfully.")

In [ ]:
# STEP 51.18 — Test complete Feed AI

sample_index = 9

sample_spectrum = X_ivdmd_test[sample_index]

result = feed_ai_predict_v2(sample_spectrum)

print("========================================")
print("       COMPLETE FEED AI RESULT")
print("========================================")

print("\nNUTRITIONAL PREDICTIONS")
print("------------------------")

for nutrient, value in result["nutritional_predictions"].items():
    print(f"{nutrient}: {value}")

print("\nSPECTRAL ANOMALY")
print("----------------")

for key, value in result["spectral_anomaly"].items():
    print(f"{key}: {value}")

print("\nQUALITY ASSESSMENT")
print("------------------")

print(result["quality_assessment"])

In [ ]:
# STEP 52.1 FIX — Rebuild test targets and validate all 4 models

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------------------------------------------
# 1. Load the original FullData dataset
# ---------------------------------------------------------

full_data_path = "/content/02a._Fulldata[1].tab"

df_validation = pd.read_csv(
    full_data_path,
    sep="\t",
    low_memory=False
)

print("Full dataset shape:", df_validation.shape)

# ---------------------------------------------------------
# 2. Identify the 1050 NIR wavelength columns
# ---------------------------------------------------------

wavelength_cols = [
    col for col in df_validation.columns
    if str(col).replace(".", "", 1).isdigit()
]

wavelength_cols = sorted(
    wavelength_cols,
    key=lambda x: float(x)
)

print("Number of NIR wavelengths:", len(wavelength_cols))

# ---------------------------------------------------------
# 3. Function to prepare one target
# ---------------------------------------------------------

def rebuild_target_test_data(df, target_column):

    temp = df[[target_column] + wavelength_cols].copy()

    # Convert target and spectral columns to numeric
    temp[target_column] = pd.to_numeric(
        temp[target_column],
        errors="coerce"
    )

    for col in wavelength_cols:
        temp[col] = pd.to_numeric(
            temp[col],
            errors="coerce"
        )

    # Remove rows with missing target or NIR values
    temp = temp.dropna(
        subset=[target_column] + wavelength_cols
    )

    X = temp[wavelength_cols].values
    y = temp[target_column].values

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    return X_train, X_test, y_train, y_test


# ---------------------------------------------------------
# 4. Rebuild the four target test sets
# ---------------------------------------------------------

X_cp_train, X_cp_test, y_cp_train, y_cp_test = \
    rebuild_target_test_data(df_validation, "PC")

X_ndf_train, X_ndf_test, y_ndf_train, y_ndf_test = \
    rebuild_target_test_data(df_validation, "FDN")

X_adf_train, X_adf_test, y_adf_train, y_adf_test = \
    rebuild_target_test_data(df_validation, "FDA")

X_ivdmd_train_check, X_ivdmd_test_check, y_ivdmd_train_check, y_ivdmd_test_check = \
    rebuild_target_test_data(df_validation, "DIVMS")


print("\nTEST SET SIZES")
print("----------------")
print("CP:", len(y_cp_test))
print("NDF:", len(y_ndf_test))
print("ADF:", len(y_adf_test))
print("IVDMD:", len(y_ivdmd_test))


# ---------------------------------------------------------
# 5. Generate predictions using the corresponding test sets
# ---------------------------------------------------------

cp_pred = final_cp_model.predict(X_cp_test).ravel()
ndf_pred = final_ndf_model.predict(X_ndf_test).ravel()
adf_pred = final_adf_model.predict(X_adf_test).ravel()
ivdmd_pred = final_ivdmd_model.predict(X_ivdmd_test_check).ravel()


# ---------------------------------------------------------
# 6. Calculate validation metrics
# ---------------------------------------------------------

targets = {
    "CP": (y_cp_test, cp_pred),
    "NDF": (y_ndf_test, ndf_pred),
    "ADF": (y_adf_test, adf_pred),
    "IVDMD": (y_ivdmd_test_check, ivdmd_pred)
}

validation_results = []

for nutrient, (actual, predicted) in targets.items():

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = r2_score(actual, predicted)

    validation_results.append({
        "Nutrient": nutrient,
        "Test Samples": len(actual),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "R2 (%)": r2 * 100
    })


validation_df = pd.DataFrame(validation_results)


# ---------------------------------------------------------
# 7. Display final validation table
# ---------------------------------------------------------

print("\n==============================================")
print("       COMPLETE NUTRITIONAL VALIDATION")
print("==============================================")

print(
    validation_df.round(4).to_string(index=False)
)

In [ ]:
# STEP 52.2 — Prediction error analysis for all four nutrients

error_analysis = {}

for nutrient, (actual, predicted) in targets.items():

    absolute_error = np.abs(actual - predicted)

    error_analysis[nutrient] = {
        "mean_error": np.mean(absolute_error),
        "median_error": np.median(absolute_error),
        "max_error": np.max(absolute_error),
        "std_error": np.std(absolute_error)
    }

    print("\n" + "=" * 45)
    print(f"{nutrient} ERROR ANALYSIS")
    print("=" * 45)

    print(f"Mean Absolute Error   : {np.mean(absolute_error):.4f}")
    print(f"Median Absolute Error : {np.median(absolute_error):.4f}")
    print(f"Maximum Error         : {np.max(absolute_error):.4f}")
    print(f"Error Std. Deviation  : {np.std(absolute_error):.4f}")

    # Top 5 largest errors
    largest_indices = np.argsort(absolute_error)[-5:][::-1]

    print("\nTop 5 largest prediction errors:")

    for rank, idx in enumerate(largest_indices, start=1):

        print(
            f"{rank}. "
            f"Test index: {idx} | "
            f"Actual: {actual[idx]:.3f} | "
            f"Predicted: {predicted[idx]:.3f} | "
            f"Error: {absolute_error[idx]:.3f}"
        )

In [ ]:
# STEP 52.3 — Anomaly status vs IVDMD prediction error

import numpy as np
import pandas as pd

ivdmd_actual = y_ivdmd_test
ivdmd_predicted = final_ivdmd_model.predict(X_ivdmd_test).ravel()

records = []

for i in range(len(X_ivdmd_test)):

    spectrum = X_ivdmd_test[i]

    anomaly_result = nir_anomaly_detector(spectrum)

    actual = float(ivdmd_actual[i])
    predicted = float(ivdmd_predicted[i])
    error = abs(actual - predicted)

    records.append({
        "Test_Index": i,
        "Actual_IVDMD": actual,
        "Predicted_IVDMD": predicted,
        "Absolute_Error": error,
        "Anomaly_Status": anomaly_result["combined_status"],
        "PCA_Distance": anomaly_result["pca_distance"],
        "Isolation_Forest_Score":
            anomaly_result["isolation_forest_score"]
    })

ivdmd_error_df = pd.DataFrame(records)

# --------------------------------------------------
# Separate normal and anomalous samples
# --------------------------------------------------

normal_mask = (
    ivdmd_error_df["Anomaly_Status"] == "NORMAL"
)

anomalous_mask = (
    ivdmd_error_df["Anomaly_Status"] != "NORMAL"
)

normal_errors = ivdmd_error_df.loc[
    normal_mask, "Absolute_Error"
]

anomalous_errors = ivdmd_error_df.loc[
    anomalous_mask, "Absolute_Error"
]

print("==============================================")
print("   ANOMALY vs IVDMD PREDICTION ERROR")
print("==============================================")

print("\nNORMAL SAMPLES")
print("----------------")
print("Count:", len(normal_errors))

if len(normal_errors) > 0:
    print(
        "Mean absolute error:",
        round(normal_errors.mean(), 4)
    )
    print(
        "Median absolute error:",
        round(normal_errors.median(), 4)
    )
    print(
        "Maximum error:",
        round(normal_errors.max(), 4)
    )

print("\nANOMALOUS SAMPLES")
print("------------------")
print("Count:", len(anomalous_errors))

if len(anomalous_errors) > 0:
    print(
        "Mean absolute error:",
        round(anomalous_errors.mean(), 4)
    )
    print(
        "Median absolute error:",
        round(anomalous_errors.median(), 4)
    )
    print(
        "Maximum error:",
        round(anomalous_errors.max(), 4)
    )

# --------------------------------------------------
# Largest errors with anomaly status
# --------------------------------------------------

print("\nTOP 10 IVDMD ERRORS WITH ANOMALY STATUS")
print("------------------------------------------")

top_errors = ivdmd_error_df.sort_values(
    "Absolute_Error",
    ascending=False
).head(10)

for _, row in top_errors.iterrows():

    print(
        f"Index {int(row['Test_Index'])} | "
        f"Actual: {row['Actual_IVDMD']:.2f} | "
        f"Predicted: {row['Predicted_IVDMD']:.2f} | "
        f"Error: {row['Absolute_Error']:.2f} | "
        f"Status: {row['Anomaly_Status']} | "
        f"PCA distance: {row['PCA_Distance']:.2f}"
    )

In [ ]:
# STEP 52.3 FIX — Correct anomaly vs IVDMD prediction error analysis

import numpy as np
import pandas as pd

ivdmd_actual = y_ivdmd_test
ivdmd_predicted = final_ivdmd_model.predict(X_ivdmd_test).ravel()

records = []

for i in range(len(X_ivdmd_test)):

    spectrum = X_ivdmd_test[i]

    anomaly_result = nir_anomaly_detector(spectrum)

    actual = float(ivdmd_actual[i])
    predicted = float(ivdmd_predicted[i])
    error = abs(actual - predicted)

    combined_status = anomaly_result["combined_status"]

    # Correct interpretation
    if combined_status == "STRONG_SPECTRAL_WARNING":
        anomaly_group = "ANOMALOUS"
    else:
        anomaly_group = "NORMAL"

    records.append({
        "Test_Index": i,
        "Actual_IVDMD": actual,
        "Predicted_IVDMD": predicted,
        "Absolute_Error": error,
        "Anomaly_Group": anomaly_group,
        "Combined_Status": combined_status,
        "PCA_Distance": anomaly_result["pca_distance"],
        "Isolation_Forest_Score":
            anomaly_result["isolation_forest_score"]
    })

ivdmd_error_df = pd.DataFrame(records)

# --------------------------------------------------
# Separate groups
# --------------------------------------------------

normal_errors = ivdmd_error_df.loc[
    ivdmd_error_df["Anomaly_Group"] == "NORMAL",
    "Absolute_Error"
]

anomalous_errors = ivdmd_error_df.loc[
    ivdmd_error_df["Anomaly_Group"] == "ANOMALOUS",
    "Absolute_Error"
]

print("==============================================")
print("   CORRECTED ANOMALY vs IVDMD ERROR ANALYSIS")
print("==============================================")

print("\nNORMAL SAMPLES")
print("----------------")
print("Count:", len(normal_errors))

if len(normal_errors) > 0:
    print("Mean absolute error:",
          round(normal_errors.mean(), 4))
    print("Median absolute error:",
          round(normal_errors.median(), 4))
    print("Maximum error:",
          round(normal_errors.max(), 4))

print("\nANOMALOUS SAMPLES")
print("------------------")
print("Count:", len(anomalous_errors))

if len(anomalous_errors) > 0:
    print("Mean absolute error:",
          round(anomalous_errors.mean(), 4))
    print("Median absolute error:",
          round(anomalous_errors.median(), 4))
    print("Maximum error:",
          round(anomalous_errors.max(), 4))

# --------------------------------------------------
# Compare mean errors
# --------------------------------------------------

if len(normal_errors) > 0 and len(anomalous_errors) > 0:

    print("\nERROR COMPARISON")
    print("----------------")

    print(
        "Anomalous / Normal mean-error ratio:",
        round(
            anomalous_errors.mean() /
            normal_errors.mean(),
            3
        )
    )

# --------------------------------------------------
# Largest prediction errors
# --------------------------------------------------

print("\nTOP 10 IVDMD ERRORS")
print("--------------------")

top_errors = ivdmd_error_df.sort_values(
    "Absolute_Error",
    ascending=False
).head(10)

for _, row in top_errors.iterrows():

    print(
        f"Index {int(row['Test_Index'])} | "
        f"Actual: {row['Actual_IVDMD']:.2f} | "
        f"Predicted: {row['Predicted_IVDMD']:.2f} | "
        f"Error: {row['Absolute_Error']:.2f} | "
        f"Group: {row['Anomaly_Group']} | "
        f"PCA distance: {row['PCA_Distance']:.2f}"
    )

In [ ]:
# STEP 52.4 — Inspect anomalous samples

anomalous_samples = ivdmd_error_df[
    ivdmd_error_df["Anomaly_Group"] == "ANOMALOUS"
].copy()

print("==============================================")
print("        ANOMALOUS SAMPLE INSPECTION")
print("==============================================")

print("\nNumber of anomalous samples:",
      len(anomalous_samples))

print("\nAnomalous samples:")
print("------------------")

for _, row in anomalous_samples.iterrows():

    print(
        f"Index {int(row['Test_Index'])} | "
        f"IVDMD Actual: {row['Actual_IVDMD']:.2f} | "
        f"Predicted: {row['Predicted_IVDMD']:.2f} | "
        f"Error: {row['Absolute_Error']:.2f} | "
        f"PCA distance: {row['PCA_Distance']:.2f} | "
        f"IF score: {row['Isolation_Forest_Score']:.4f}"
    )

print("\n==============================================")
print("ANOMALOUS SAMPLE ERROR SUMMARY")
print("==============================================")

if len(anomalous_samples) > 0:

    print(
        "Mean error:",
        round(
            anomalous_samples["Absolute_Error"].mean(),
            4
        )
    )

    print(
        "Median error:",
        round(
            anomalous_samples["Absolute_Error"].median(),
            4
        )
    )

    print(
        "Maximum error:",
        round(
            anomalous_samples["Absolute_Error"].max(),
            4
        )
    )

In [ ]:
# STEP 53.1 — Batch-level NIR nutritional prediction

def feed_ai_batch_predict(nir_batch):
    """
    Predict CP, NDF, ADF and IVDMD for a batch
    of NIR spectra.

    Input:
        nir_batch -> 2D array
                      shape = (number_of_samples, 1050)

    Output:
        Nutritional predictions for every sample.
    """

    # Convert to numpy array
    batch = np.asarray(nir_batch)

    # Make sure input is 2D
    if batch.ndim != 2:
        raise ValueError(
            "Input must be a 2D array: "
            "(number_of_samples, 1050)"
        )

    # Check wavelength count
    expected_features = X_anomaly_train.shape[1]

    if batch.shape[1] != expected_features:
        raise ValueError(
            f"Expected {expected_features} NIR values per sample, "
            f"but received {batch.shape[1]}"
        )

    # ---------------------------------------------
    # Predict all four nutritional components
    # ---------------------------------------------

    cp_predictions = (
        final_cp_model.predict(batch)
        .ravel()
    )

    ndf_predictions = (
        final_ndf_model.predict(batch)
        .ravel()
    )

    adf_predictions = (
        final_adf_model.predict(batch)
        .ravel()
    )

    ivdmd_predictions = (
        final_ivdmd_model.predict(batch)
        .ravel()
    )

    # ---------------------------------------------
    # Create result table
    # ---------------------------------------------

    batch_results = pd.DataFrame({
        "Sample": np.arange(1, len(batch) + 1),

        "CP": cp_predictions,

        "NDF": ndf_predictions,

        "ADF": adf_predictions,

        "IVDMD": ivdmd_predictions
    })

    return batch_results


print("Batch nutritional prediction function created successfully.")

In [ ]:
# STEP 53.2 — Test batch prediction on 10 real NIR samples

# Take 10 real spectra from the existing IVDMD test set
nir_batch_test = X_ivdmd_test[:10]

# Generate predictions for all 4 nutritional parameters
batch_results = feed_ai_batch_predict(nir_batch_test)

# Display results
print("Batch prediction completed.")
print("Number of samples:", len(batch_results))
print("Columns:", list(batch_results.columns))

display(batch_results.round(3))

In [ ]:
# STEP 53.3 — Add NIR anomaly detection to batch predictions

def feed_ai_batch_predict_v2(nir_batch):
    batch = np.asarray(nir_batch)

    if batch.ndim != 2:
        raise ValueError("nir_batch must be a 2D array")

    expected_features = X_anomaly_train.shape[1]

    if batch.shape[1] != expected_features:
        raise ValueError(
            f"Expected {expected_features} NIR features, "
            f"but received {batch.shape[1]}"
        )

    # Nutritional predictions
    results = feed_ai_batch_predict(batch).copy()

    # Anomaly detection
    anomaly_rows = []

    for spectrum in batch:
        anomaly = nir_anomaly_detector(spectrum)

        anomaly_rows.append({
            "IF_Status": anomaly["isolation_forest_status"],
            "IF_Score": anomaly["isolation_forest_score"],
            "PCA_Distance": anomaly["pca_distance"],
            "PCA_Status": anomaly["pca_distance_status"],
            "Combined_Status": anomaly["combined_status"]
        })

    anomaly_df = pd.DataFrame(anomaly_rows)

    # Combine nutritional predictions + anomaly information
    results = pd.concat(
        [results, anomaly_df],
        axis=1
    )

    return results


# Test on the same 10 real NIR samples
batch_results_v2 = feed_ai_batch_predict_v2(nir_batch_test)

print("Batch AI + anomaly analysis completed.")
print("Number of samples:", len(batch_results_v2))

display(batch_results_v2.round(3))

In [ ]:
# STEP 53.4 — Batch-level summary

def summarize_feed_batch(batch_results):
    nutrient_cols = ["CP", "NDF", "ADF", "IVDMD"]

    summary = {
        "total_samples": len(batch_results),
        "strong_spectral_warnings": int(
            (batch_results["Combined_Status"] == "STRONG_SPECTRAL_WARNING").sum()
        ),
        "warning_percentage": round(
            100 * (
                batch_results["Combined_Status"]
                == "STRONG_SPECTRAL_WARNING"
            ).mean(),
            2
        )
    }

    # Nutritional statistics
    for nutrient in nutrient_cols:
        summary[f"{nutrient}_mean"] = batch_results[nutrient].mean()
        summary[f"{nutrient}_min"] = batch_results[nutrient].min()
        summary[f"{nutrient}_max"] = batch_results[nutrient].max()

    return summary


batch_summary = summarize_feed_batch(batch_results_v2)

print("BATCH SUMMARY")
print("=" * 50)

for key, value in batch_summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.3f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# STEP 57.1 — AI Decision Engine

def ai_decision_engine(predictions, spectral_anomaly=None):
    """
    Convert nutritional predictions + spectral anomaly information
    into a structured quality/risk/advisory result.
    """

    cp = float(predictions["CP"])
    ndf = float(predictions["NDF"])
    adf = float(predictions["ADF"])
    ivdmd = float(predictions["IVDMD"])

    # -----------------------------
    # Nutritional status
    # -----------------------------

    if cp < 7:
        cp_status = "LOW"
    elif cp <= 12:
        cp_status = "ADEQUATE"
    else:
        cp_status = "HIGH"

    if ndf > 60:
        ndf_status = "HIGH"
    elif ndf >= 40:
        ndf_status = "MODERATE"
    else:
        ndf_status = "LOW"

    if adf > 40:
        adf_status = "HIGH"
    elif adf >= 30:
        adf_status = "MODERATE"
    else:
        adf_status = "LOW"

    if ivdmd < 55:
        ivdmd_status = "LOW"
    elif ivdmd <= 65:
        ivdmd_status = "MODERATE"
    else:
        ivdmd_status = "GOOD"

    individual_status = {
        "CP": cp_status,
        "NDF": ndf_status,
        "ADF": adf_status,
        "IVDMD": ivdmd_status
    }

    # -----------------------------
    # Risk flags
    # -----------------------------

    risk_flags = []

    if cp_status == "LOW":
        risk_flags.append("LOW_PROTEIN")

    if ndf_status == "HIGH":
        risk_flags.append("HIGH_FIBER_NDF")

    if adf_status == "HIGH":
        risk_flags.append("HIGH_FIBER_ADF")

    if ivdmd_status == "LOW":
        risk_flags.append("LOW_DIGESTIBILITY")

    # -----------------------------
    # Spectral warning
    # -----------------------------

    spectral_warning = False

    if spectral_anomaly is not None:
        if spectral_anomaly.get("combined_status") == "STRONG_SPECTRAL_WARNING":
            spectral_warning = True
            risk_flags.append("UNUSUAL_SPECTRAL_PATTERN")

    # -----------------------------
    # Overall quality decision
    # -----------------------------

    severe_risks = sum([
        cp_status == "LOW",
        ndf_status == "HIGH",
        adf_status == "HIGH",
        ivdmd_status == "LOW"
    ])

    if severe_risks >= 2:
        quality_status = "NEEDS_ATTENTION"
    elif severe_risks == 1:
        quality_status = "MODERATE"
    else:
        quality_status = "GOOD"

    # Spectral warning does not automatically mean bad feed.
    # It means the sample deserves additional attention.
    if spectral_warning and quality_status == "GOOD":
        quality_status = "REVIEW_REQUIRED"

    # -----------------------------
    # Recommendations
    # -----------------------------

    recommendations = []

    if cp_status == "LOW":
        recommendations.append(
            "Consider protein supplementation or a higher-protein feed source."
        )

    if ndf_status == "HIGH":
        recommendations.append(
            "Review forage maturity, fiber level, and expected feed intake."
        )

    if adf_status == "HIGH":
        recommendations.append(
            "Review forage quality and processing because high ADF may reduce digestibility."
        )

    if ivdmd_status == "LOW":
        recommendations.append(
            "Low predicted digestibility; review forage maturity, processing, and ration formulation."
        )

    if spectral_warning:
        recommendations.append(
            "Unusual spectral pattern detected; verify the sample or repeat measurement before making a final decision."
        )

    if not recommendations:
        recommendations.append(
            "Nutritional indicators are within the prototype reference ranges."
        )

    return {
        "quality_status": quality_status,
        "individual_status": individual_status,
        "risk_flags": risk_flags,
        "spectral_warning": spectral_warning,
        "recommendations": recommendations
    }


print("AI Decision Engine created successfully.")

In [ ]:
# STEP 57.2 — Test Decision Engine on Sample 10

sample_index = 9

# Get nutritional predictions
sample_predictions = {
    "CP": batch_results_v2.loc[sample_index, "CP"],
    "NDF": batch_results_v2.loc[sample_index, "NDF"],
    "ADF": batch_results_v2.loc[sample_index, "ADF"],
    "IVDMD": batch_results_v2.loc[sample_index, "IVDMD"]
}

# Get anomaly information
sample_anomaly = {
    "combined_status": batch_results_v2.loc[sample_index, "Combined_Status"],
    "isolation_forest_status": batch_results_v2.loc[sample_index, "IF_Status"],
    "isolation_forest_score": batch_results_v2.loc[sample_index, "IF_Score"],
    "pca_distance": batch_results_v2.loc[sample_index, "PCA_Distance"],
    "pca_distance_status": batch_results_v2.loc[sample_index, "PCA_Status"]
}

# Run decision engine
sample_decision = ai_decision_engine(
    sample_predictions,
    sample_anomaly
)

print("AI DECISION ENGINE RESULT")
print("=" * 50)

print("Predictions:")
for key, value in sample_predictions.items():
    print(f"  {key}: {value:.3f}")

print("\nQuality Status:")
print(" ", sample_decision["quality_status"])

print("\nIndividual Status:")
for key, value in sample_decision["individual_status"].items():
    print(f"  {key}: {value}")

print("\nRisk Flags:")
for risk in sample_decision["risk_flags"]:
    print(" ", risk)

print("\nRecommendations:")
for recommendation in sample_decision["recommendations"]:
    print(" ", recommendation)

In [ ]:
# STEP 57.3 — Apply AI Decision Engine to the complete batch

batch_decisions = []

for i in range(len(batch_results_v2)):

    # Nutritional predictions
    predictions = {
        "CP": batch_results_v2.loc[i, "CP"],
        "NDF": batch_results_v2.loc[i, "NDF"],
        "ADF": batch_results_v2.loc[i, "ADF"],
        "IVDMD": batch_results_v2.loc[i, "IVDMD"]
    }

    # Spectral anomaly information
    anomaly = {
        "combined_status": batch_results_v2.loc[i, "Combined_Status"],
        "isolation_forest_status": batch_results_v2.loc[i, "IF_Status"],
        "isolation_forest_score": batch_results_v2.loc[i, "IF_Score"],
        "pca_distance": batch_results_v2.loc[i, "PCA_Distance"],
        "pca_distance_status": batch_results_v2.loc[i, "PCA_Status"]
    }

    # Decision engine
    decision = ai_decision_engine(
        predictions,
        anomaly
    )

    batch_decisions.append({
        "Sample": batch_results_v2.loc[i, "Sample"],
        "Quality_Status": decision["quality_status"],
        "Risk_Count": len(decision["risk_flags"]),
        "Risk_Flags": ", ".join(decision["risk_flags"]),
        "Spectral_Warning": decision["spectral_warning"],
        "Recommendations": " | ".join(decision["recommendations"])
    })


batch_decision_df = pd.DataFrame(batch_decisions)

print("Batch decision engine completed.")
print("Samples processed:", len(batch_decision_df))

display(batch_decision_df)

In [ ]:
# STEP 57.4 — Summarize batch decisions

quality_counts = batch_decision_df["Quality_Status"].value_counts()

spectral_warning_count = int(
    batch_decision_df["Spectral_Warning"].sum()
)

print("BATCH AI DECISION SUMMARY")
print("=" * 50)

print("Total samples:", len(batch_decision_df))

print("\nQuality distribution:")
for status, count in quality_counts.items():
    percentage = (count / len(batch_decision_df)) * 100
    print(f"  {status}: {count} ({percentage:.1f}%)")

print("\nStrong spectral warnings:")
print(
    f"  {spectral_warning_count} "
    f"({spectral_warning_count / len(batch_decision_df) * 100:.1f}%)"
)

print("\nSamples requiring attention:")
attention_count = int(
    (batch_decision_df["Quality_Status"] == "NEEDS_ATTENTION").sum()
)

print(
    f"  {attention_count} "
    f"({attention_count / len(batch_decision_df) * 100:.1f}%)"
)

In [ ]:
# STEP 57.5 — Final single-sample AI pipeline

def final_feed_analysis(nir_spectrum):

    spectrum = np.asarray(nir_spectrum)

    # Check input shape
    if spectrum.ndim != 1:
        raise ValueError("Input NIR spectrum must be a 1D array.")

    expected_features = X_anomaly_train.shape[1]

    if len(spectrum) != expected_features:
        raise ValueError(
            f"Expected {expected_features} NIR values, "
            f"but received {len(spectrum)}."
        )

    # --------------------------------
    # 1. Nutritional prediction
    # --------------------------------

    predictions = {
        "CP": float(final_cp_model.predict(spectrum.reshape(1, -1)).ravel()[0]),
        "NDF": float(final_ndf_model.predict(spectrum.reshape(1, -1)).ravel()[0]),
        "ADF": float(final_adf_model.predict(spectrum.reshape(1, -1)).ravel()[0]),
        "IVDMD": float(final_ivdmd_model.predict(spectrum.reshape(1, -1)).ravel()[0])
    }

    # --------------------------------
    # 2. Spectral anomaly detection
    # --------------------------------

    anomaly = nir_anomaly_detector(spectrum)

    # --------------------------------
    # 3. AI Decision Engine
    # --------------------------------

    decision = ai_decision_engine(
        predictions,
        anomaly
    )

    # --------------------------------
    # 4. Final combined result
    # --------------------------------

    return {
        "nutritional_predictions": predictions,
        "spectral_anomaly": anomaly,
        "quality_assessment": decision
    }


print("Final single-sample AI pipeline created successfully.")

In [ ]:
# STEP 57.6 — Test final AI pipeline on one real NIR spectrum

# Select one real NIR spectrum
test_spectrum = X_ivdmd_test[9]

# Run the complete AI pipeline
final_result = final_feed_analysis(test_spectrum)

# Display nutritional predictions
print("FINAL AI ANALYSIS")
print("=" * 60)

print("\nNutritional Predictions:")
for nutrient, value in final_result["nutritional_predictions"].items():
    print(f"  {nutrient}: {value:.3f}")

# Display anomaly result
anomaly = final_result["spectral_anomaly"]

print("\nSpectral Anomaly:")
print(f"  Isolation Forest: {anomaly['isolation_forest_status']}")
print(f"  IF Score: {anomaly['isolation_forest_score']:.4f}")
print(f"  PCA Distance: {anomaly['pca_distance']:.3f}")
print(f"  PCA Status: {anomaly['pca_distance_status']}")
print(f"  Combined Status: {anomaly['combined_status']}")

# Display decision
decision = final_result["quality_assessment"]

print("\nQuality Assessment:")
print(f"  Overall Quality: {decision['quality_status']}")

print("\nIndividual Status:")
for nutrient, status in decision["individual_status"].items():
    print(f"  {nutrient}: {status}")

print("\nRisk Flags:")
if decision["risk_flags"]:
    for risk in decision["risk_flags"]:
        print(f"  - {risk}")
else:
    print("  None")

print("\nRecommendations:")
for recommendation in decision["recommendations"]:
    print(f"  - {recommendation}")

In [ ]:
# STEP 58.1 — Create model reliability information

# Typical validation error (MAE) from our held-out test sets
model_reliability = {
    "CP": {
        "MAE": 0.4685,
        "error_unit": "%"
    },
    "NDF": {
        "MAE": 1.1320,
        "error_unit": "%"
    },
    "ADF": {
        "MAE": 0.8613,
        "error_unit": "%"
    },
    "IVDMD": {
        "MAE": 2.0778,
        "error_unit": "%"
    }
}

print("MODEL RELIABILITY")
print("=" * 50)

for nutrient, info in model_reliability.items():
    print(
        f"{nutrient}: Typical validation error "
        f"± {info['MAE']:.4f} {info['error_unit']}"
    )

In [ ]:
# STEP 58.2 — Add model reliability to final AI analysis

def final_feed_analysis_v2(nir_spectrum):

    spectrum = np.asarray(nir_spectrum)

    # Check input shape
    if spectrum.ndim != 1:
        raise ValueError("Input NIR spectrum must be a 1D array.")

    expected_features = X_anomaly_train.shape[1]

    if len(spectrum) != expected_features:
        raise ValueError(
            f"Expected {expected_features} NIR values, "
            f"but received {len(spectrum)}."
        )

    # --------------------------------
    # 1. Nutritional prediction
    # --------------------------------

    predictions = {
        "CP": float(
            final_cp_model.predict(
                spectrum.reshape(1, -1)
            ).ravel()[0]
        ),
        "NDF": float(
            final_ndf_model.predict(
                spectrum.reshape(1, -1)
            ).ravel()[0]
        ),
        "ADF": float(
            final_adf_model.predict(
                spectrum.reshape(1, -1)
            ).ravel()[0]
        ),
        "IVDMD": float(
            final_ivdmd_model.predict(
                spectrum.reshape(1, -1)
            ).ravel()[0]
        )
    }

    # --------------------------------
    # 2. Reliability information
    # --------------------------------

    reliability = {}

    for nutrient, value in predictions.items():

        error = model_reliability[nutrient]["MAE"]

        reliability[nutrient] = {
            "prediction": value,
            "typical_validation_error": error,
            "lower_reference": value - error,
            "upper_reference": value + error
        }

    # --------------------------------
    # 3. Spectral anomaly detection
    # --------------------------------

    anomaly = nir_anomaly_detector(spectrum)

    # --------------------------------
    # 4. AI Decision Engine
    # --------------------------------

    decision = ai_decision_engine(
        predictions,
        anomaly
    )

    # --------------------------------
    # 5. Final combined result
    # --------------------------------

    return {
        "nutritional_predictions": predictions,
        "prediction_reliability": reliability,
        "spectral_anomaly": anomaly,
        "quality_assessment": decision
    }


print("Final AI pipeline v2 with reliability created successfully.")

In [ ]:
# STEP 58.3 — Test final AI pipeline v2

test_spectrum = X_ivdmd_test[9]

result_v2 = final_feed_analysis_v2(test_spectrum)

print("FINAL AI ANALYSIS v2")
print("=" * 60)

print("\nNutritional Predictions + Typical Validation Error:")

for nutrient, info in result_v2["prediction_reliability"].items():
    print(
        f"  {nutrient}: "
        f"{info['prediction']:.3f} "
        f"± {info['typical_validation_error']:.3f}"
    )

print("\nSpectral Anomaly:")
anomaly = result_v2["spectral_anomaly"]

print(f"  Isolation Forest: {anomaly['isolation_forest_status']}")
print(f"  IF Score: {anomaly['isolation_forest_score']:.4f}")
print(f"  PCA Distance: {anomaly['pca_distance']:.3f}")
print(f"  PCA Status: {anomaly['pca_distance_status']}")
print(f"  Combined Status: {anomaly['combined_status']}")

print("\nQuality Assessment:")
decision = result_v2["quality_assessment"]

print(f"  Overall Quality: {decision['quality_status']}")

print("\nRisk Flags:")
for risk in decision["risk_flags"]:
    print(f"  - {risk}")

print("\nRecommendations:")
for recommendation in decision["recommendations"]:
    print(f"  - {recommendation}")

In [ ]:
# STEP 58.4 — Create a reliability summary for the final result

def reliability_summary(result):

    reliability = result["prediction_reliability"]
    anomaly = result["spectral_anomaly"]

    summary = {}

    # Typical validation error for each prediction
    for nutrient, info in reliability.items():
        summary[nutrient] = {
            "prediction": round(info["prediction"], 3),
            "typical_validation_error": round(
                info["typical_validation_error"], 3
            )
        }

    # Overall reliability warning
    if anomaly["combined_status"] == "STRONG_SPECTRAL_WARNING":
        overall_status = "REVIEW_REQUIRED"
        message = (
            "The spectrum is unusual compared with the model training data. "
            "Predictions should be verified with a repeat measurement or "
            "laboratory analysis before making a final feed-quality decision."
        )
    else:
        overall_status = "NORMAL_INPUT_RANGE"
        message = (
            "The spectrum is within the model's observed spectral range. "
            "Predictions should still be interpreted using the reported "
            "typical validation errors."
        )

    summary["overall_reliability_status"] = overall_status
    summary["message"] = message

    return summary


reliability_result = reliability_summary(result_v2)

print("RELIABILITY SUMMARY")
print("=" * 60)

for nutrient in ["CP", "NDF", "ADF", "IVDMD"]:
    info = reliability_result[nutrient]

    print(
        f"{nutrient}: "
        f"{info['prediction']:.3f} "
        f"± {info['typical_validation_error']:.3f}"
    )

print("\nOverall Reliability Status:")
print(" ", reliability_result["overall_reliability_status"])

print("\nMessage:")
print(" ", reliability_result["message"])

In [ ]:
# STEP 59.1 — Create deployment-ready result structure

def create_deployment_result(
    result,
    sample_id="SAMPLE_001"
):

    predictions = result["nutritional_predictions"]
    reliability = result["prediction_reliability"]
    anomaly = result["spectral_anomaly"]
    quality = result["quality_assessment"]

    deployment_result = {
        "sample_id": sample_id,

        "nutritional_analysis": {
            "CP_percent": round(predictions["CP"], 3),
            "NDF_percent": round(predictions["NDF"], 3),
            "ADF_percent": round(predictions["ADF"], 3),
            "IVDMD_percent": round(predictions["IVDMD"], 3)
        },

        "prediction_reliability": {
            nutrient: {
                "prediction": round(info["prediction"], 3),
                "typical_validation_error": round(
                    info["typical_validation_error"], 3
                )
            }
            for nutrient, info in reliability.items()
        },

        "spectral_analysis": {
            "isolation_forest_status":
                anomaly["isolation_forest_status"],

            "isolation_forest_score":
                round(anomaly["isolation_forest_score"], 4),

            "pca_distance":
                round(anomaly["pca_distance"], 3),

            "pca_status":
                anomaly["pca_distance_status"],

            "combined_status":
                anomaly["combined_status"]
        },

        "quality_assessment": {
            "overall_status":
                quality["quality_status"],

            "individual_status":
                quality["individual_status"],

            "risk_flags":
                quality["risk_flags"],

            "recommendations":
                quality["recommendations"]
        }
    }

    return deployment_result


# Create deployment-ready result for our real Sample 10
deployment_result = create_deployment_result(
    result_v2,
    sample_id="TEST_SAMPLE_10"
)

print("Deployment-ready result created successfully.")
print(deployment_result)

In [ ]:
# STEP 59.2 — Validate deployment-ready output structure

required_sections = [
    "sample_id",
    "nutritional_analysis",
    "prediction_reliability",
    "spectral_analysis",
    "quality_assessment"
]

required_nutrients = [
    "CP_percent",
    "NDF_percent",
    "ADF_percent",
    "IVDMD_percent"
]

missing_sections = [
    section
    for section in required_sections
    if section not in deployment_result
]

missing_nutrients = [
    nutrient
    for nutrient in required_nutrients
    if nutrient not in deployment_result["nutritional_analysis"]
]

print("DEPLOYMENT STRUCTURE VALIDATION")
print("=" * 60)

if not missing_sections and not missing_nutrients:
    print("STATUS: PASS")
    print("All required sections are present.")
    print("All 4 nutritional predictions are present.")
else:
    print("STATUS: FAIL")

    if missing_sections:
        print("Missing sections:", missing_sections)

    if missing_nutrients:
        print("Missing nutritional fields:", missing_nutrients)

In [ ]:
import json

# Generate a clean deployment-ready result
deployment_result = create_deployment_result(
    final_feed_analysis_v2(X_ivdmd_test[9]),
    sample_id="DEMO_SAMPLE_001"
)

# Convert NumPy values to standard Python types if needed
deployment_json = json.dumps(
    deployment_result,
    indent=2,
    default=lambda x: x.item() if hasattr(x, "item") else x
)

print("FINAL DEPLOYMENT JSON")
print("=" * 60)
print(deployment_json)

In [ ]:
# Save the final deployment result as a JSON file

deployment_json_path = "/content/DEMO_SAMPLE_001_result.json"

with open(deployment_json_path, "w") as f:
    f.write(deployment_json)

print("DEPLOYMENT JSON SAVED")
print("=" * 60)
print(deployment_json_path)

# Verify the file can be read back
with open(deployment_json_path, "r") as f:
    verified_result = json.load(f)

print("\nVERIFICATION: PASS")
print("Sample ID:", verified_result["sample_id"])
print("Quality Status:", verified_result["quality_assessment"]["overall_status"])
print("Spectral Status:", verified_result["spectral_analysis"]["combined_status"])

In [ ]:
def run_feed_ai_pipeline(nir_spectrum, sample_id="SAMPLE_001"):
    """
    Complete AI pipeline:
    NIR spectrum
        ↓
    Nutritional prediction
        ↓
    Reliability estimation
        ↓
    Spectral anomaly detection
        ↓
    Quality/risk decision
        ↓
    Deployment-ready JSON
    """

    # Run complete analysis
    analysis_result = final_feed_analysis_v2(nir_spectrum)

    # Convert to deployment format
    deployment_result = create_deployment_result(
        analysis_result,
        sample_id=sample_id
    )

    return deployment_result


# Test the complete pipeline
pipeline_test = run_feed_ai_pipeline(
    X_ivdmd_test[9],
    sample_id="PIPELINE_TEST_001"
)

print("COMPLETE AI PIPELINE TEST")
print("=" * 60)
print(json.dumps(
    pipeline_test,
    indent=2,
    default=lambda x: x.item() if hasattr(x, "item") else x
))

In [ ]:
print("INPUT VALIDATION TEST")
print("=" * 60)

expected_features = X_anomaly_train.shape[1]

# Test 1: Correct input
try:
    valid_test = run_feed_ai_pipeline(
        X_ivdmd_test[0],
        sample_id="VALID_TEST"
    )
    print("TEST 1 - Correct spectrum:", "PASS")
except Exception as e:
    print("TEST 1 - Correct spectrum:", "FAIL")
    print("Error:", e)


# Test 2: Wrong number of wavelengths
try:
    invalid_spectrum = X_ivdmd_test[0][:100]

    run_feed_ai_pipeline(
        invalid_spectrum,
        sample_id="INVALID_TEST"
    )

    print("TEST 2 - Wrong feature count: FAIL")
    print("Expected an input validation error, but none occurred.")

except ValueError as e:
    print("TEST 2 - Wrong feature count: PASS")
    print("Validation message:", e)


print("\nExpected wavelength features:", expected_features)
print("Validation testing complete.")

In [ ]:
import os
import json
import shutil

# Create final deployment package directory
final_package_dir = "/content/feed_ai_final_package"

os.makedirs(final_package_dir, exist_ok=True)

# Copy all required model files
required_files = [
    "/content/feed_ai_deployment/cp_pls_model.pkl",
    "/content/feed_ai_deployment/ndf_pls_model.pkl",
    "/content/feed_ai_deployment/adf_pls_model.pkl",
    "/content/feed_ai_deployment/ivdmd_pls_model.pkl",
    "/content/feed_ai_deployment/confidence_scaler.pkl",
    "/content/feed_ai_deployment/confidence_pca.pkl",
    "/content/feed_ai_deployment/confidence_threshold.pkl",
    "/content/feed_ai_deployment/wavelength_cols.pkl",
    "/content/nir_anomaly_detection/anomaly_scaler.pkl",
    "/content/nir_anomaly_detection/anomaly_pca.pkl",
    "/content/nir_anomaly_detection/isolation_forest.pkl",
    "/content/nir_anomaly_detection/pca_training_center.pkl",
    "/content/nir_anomaly_detection/pca_distance_threshold.pkl"
]

for file_path in required_files:
    if os.path.exists(file_path):
        shutil.copy(file_path, final_package_dir)
    else:
        print("WARNING - Missing:", file_path)


# Create deployment metadata
metadata = {
    "project": "Portable Cattle Feed and Silage AI Quality Assessment",
    "model_type": "PLS Regression + Spectral Anomaly Detection",
    "nutritional_targets": [
        "CP",
        "NDF",
        "ADF",
        "IVDMD"
    ],
    "nir_features": 1050,
    "wavelength_start_nm": 400,
    "wavelength_end_nm": 2498,
    "wavelength_step_nm": 2,
    "input_format": "1050 NIR spectral values",
    "outputs": [
        "nutritional_predictions",
        "prediction_reliability",
        "spectral_anomaly",
        "quality_assessment"
    ],
    "validation_r2_percent": {
        "CP": 94.41,
        "NDF": 94.01,
        "ADF": 94.30,
        "IVDMD": 83.16
    },
    "typical_validation_error": {
        "CP": 0.469,
        "NDF": 1.132,
        "ADF": 0.861,
        "IVDMD": 2.078
    },
    "important_limitation": (
        "Models were trained on Urochloa humidicola forage data. "
        "Local calibration and laboratory validation are required "
        "before real-world deployment on Indian cattle feed."
    )
}

metadata_path = os.path.join(final_package_dir, "model_metadata.json")

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)


# Create a simple deployment README
readme_text = """FEED AI FINAL DEPLOYMENT PACKAGE

Purpose:
Portable cattle feed/silage AI quality assessment.

Input:
1050 NIR spectral values from 400-2498 nm at 2 nm spacing.

Nutritional predictions:
- Crude Protein (CP)
- Neutral Detergent Fiber (NDF)
- Acid Detergent Fiber (ADF)
- In-vitro Dry Matter Digestibility (IVDMD)

AI components:
1. PLS regression models
2. Spectral anomaly detection
3. Prediction reliability estimation
4. Rule-based quality/risk decision engine

Important:
The anomaly detector identifies unusual spectral patterns.
It does NOT directly identify contamination or adulteration.

The models were trained on Urochloa humidicola forage data.
Local calibration against laboratory reference measurements is
required before real-world deployment on Indian cattle feed.

Deployment pipeline:

NIR Spectrum
    |
    v
Nutritional Models
    |
    v
Reliability + Spectral Anomaly
    |
    v
Quality/Risk Decision Engine
    |
    v
JSON API Response
"""

readme_path = os.path.join(final_package_dir, "README.txt")

with open(readme_path, "w") as f:
    f.write(readme_text)


# List final package contents
print("FINAL AI PACKAGE CREATED")
print("=" * 60)

package_files = sorted(os.listdir(final_package_dir))

for filename in package_files:
    print("✓", filename)

print("\nTotal files:", len(package_files))

In [ ]:
import os
import json

print("FINAL AI PACKAGE VALIDATION")
print("=" * 60)

required_package_files = [
    "cp_pls_model.pkl",
    "ndf_pls_model.pkl",
    "adf_pls_model.pkl",
    "ivdmd_pls_model.pkl",
    "confidence_scaler.pkl",
    "confidence_pca.pkl",
    "confidence_threshold.pkl",
    "wavelength_cols.pkl",
    "anomaly_scaler.pkl",
    "anomaly_pca.pkl",
    "isolation_forest.pkl",
    "pca_training_center.pkl",
    "pca_distance_threshold.pkl",
    "model_metadata.json",
    "README.txt"
]

missing_files = []

for filename in required_package_files:
    path = os.path.join(final_package_dir, filename)

    if os.path.exists(path):
        print(f"✓ {filename}")
    else:
        print(f"✗ MISSING: {filename}")
        missing_files.append(filename)

print("\n" + "=" * 60)

# Validate metadata
metadata_file = os.path.join(final_package_dir, "model_metadata.json")

if os.path.exists(metadata_file):
    with open(metadata_file, "r") as f:
        package_metadata = json.load(f)

    print("✓ Metadata JSON readable")
    print("✓ NIR features:", package_metadata["nir_features"])
    print("✓ Nutritional targets:", package_metadata["nutritional_targets"])

# Final status
if len(missing_files) == 0:
    print("\nSTATUS: PASS")
    print("All deployment files are present.")
else:
    print("\nSTATUS: FAIL")
    print("Missing files:", missing_files)

In [ ]:
!pip install fastapi uvicorn python-multipart nest-asyncio -q

print("FastAPI environment ready.")

In [ ]:
# Inspect all non-spectral columns in the current dataset
# to confirm exactly which laboratory targets are available.

non_spectral_cols = [
    col for col in df_validation.columns
    if col not in wavelength_cols
]

print("AVAILABLE REFERENCE / METADATA COLUMNS")
print("=" * 60)

for col in non_spectral_cols:
    print("✓", col)

print("\nTOTAL NON-SPECTRAL COLUMNS:", len(non_spectral_cols))

print("\nTARGET AVAILABILITY CHECK")
print("=" * 60)

targets_to_check = [
    "PC",       # Crude Protein
    "FDN",      # NDF
    "FDA",      # ADF
    "DIVMS",    # IVDMD
    "Moisture",
    "Water",
    "DM",
    "Urea",
    "Silica",
    "Mineral",
    "Calcium",
    "Phosphorus",
    "Magnesium"
]

for target in targets_to_check:
    matches = [c for c in df_validation.columns if target.lower() in str(c).lower()]

    if matches:
        print(f"{target:12} -> FOUND: {matches}")
    else:
        print(f"{target:12} -> NOT FOUND")

In [ ]:
# AI target registry
# Keeps track of what is currently trained and what still requires data.

ai_target_registry = {
    "CP": {
        "type": "regression",
        "status": "TRAINED",
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },
    "NDF": {
        "type": "regression",
        "status": "TRAINED",
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },
    "ADF": {
        "type": "regression",
        "status": "TRAINED",
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },
    "IVDMD": {
        "type": "regression",
        "status": "TRAINED",
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },
    "Moisture": {
        "type": "regression",
        "status": "PENDING_DATASET",
        "source": None
    },
    "Urea_Adulteration": {
        "type": "classification",
        "status": "PENDING_DATASET",
        "source": None
    },
    "Sand_Silica": {
        "type": "regression_or_classification",
        "status": "PENDING_DATASET",
        "source": None
    },
    "Mineral_Deficiency": {
        "type": "regression_or_classification",
        "status": "PENDING_DATASET",
        "source": None
    }
}

print("AI TARGET REGISTRY")
print("=" * 70)

for target, info in ai_target_registry.items():
    print(f"{target:20} | {info['type']:25} | {info['status']}")

print("\nRegistry created successfully.")

In [ ]:
import requests

print("MOISTURE DATASET SEARCH")
print("=" * 70)

urls = [
    "https://data.mendeley.com/datasets",
    "https://dataverse.harvard.edu/",
    "https://www.kaggle.com/datasets"
]

for url in urls:
    try:
        response = requests.get(url, timeout=10)
        print(f"✓ {url} -> HTTP {response.status_code}")
    except Exception as e:
        print(f"✗ {url} -> {e}")

print("\nNext step: we will use a verified public dataset containing")
print("NIR spectra + laboratory moisture/dry-matter reference values.")

## 3. Moisture Calculation

Moisture Calculation using ZENODO MOISTURE DATASET
zenodo_api = "https://zenodo.org/api/records/15838136"


In [ ]:
import requests
import os

# Verified public Zenodo dataset:
# sensAIfood cereal NIR data with reference protein and moisture

zenodo_api = "https://zenodo.org/api/records/15838136"

print("ZENODO MOISTURE DATASET")
print("=" * 70)

response = requests.get(zenodo_api, timeout=30)

print("HTTP STATUS:", response.status_code)

if not response.ok:
    raise RuntimeError(
        f"Could not access Zenodo dataset. HTTP {response.status_code}"
    )

metadata = response.json()

print("Dataset:", metadata.get("metadata", {}).get("title"))
print("Version:", metadata.get("metadata", {}).get("version"))

files = metadata.get("files", [])

print("\nAVAILABLE FILES")
print("-" * 70)

for i, file_info in enumerate(files, start=1):
    print(f"{i}. {file_info.get('key')}")
    print(f"   Size: {file_info.get('size')} bytes")

print("\nDataset metadata retrieved successfully.")

In [ ]:
import requests
import os
import zipfile

# Zenodo file download URL
download_url = (
    "https://zenodo.org/records/15838136/files/"
    "sensAIfood_Perten.zip?download=1"
)

zip_path = "/content/sensAIfood_Perten.zip"
extract_dir = "/content/sensAIfood_Perten"

print("DOWNLOADING MOISTURE DATASET")
print("=" * 70)

response = requests.get(download_url, timeout=60)

print("HTTP STATUS:", response.status_code)

if not response.ok:
    raise RuntimeError(
        f"Download failed with HTTP status {response.status_code}"
    )

with open(zip_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", zip_path)
print("File size:", os.path.getsize(zip_path), "bytes")

# Extract
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("\nEXTRACTED FILES")
print("-" * 70)

for root, dirs, files in os.walk(extract_dir):
    for filename in files:
        full_path = os.path.join(root, filename)
        print("✓", full_path)

print("\nSTATUS: PASS")
print("Dataset downloaded and extracted successfully.")

In [ ]:
import pandas as pd
import os

dataset_dir = "/content/sensAIfood_Perten"

csv_files = [
    "Barley_sensAIfood_Perten.csv",
    "Corn_sensAIfood_Perten.csv",
    "Wheat_sensAIfood_Perten.csv"
]

print("MOISTURE DATASET STRUCTURE")
print("=" * 70)

for filename in csv_files:

    path = os.path.join(dataset_dir, filename)

    df_temp = pd.read_csv(path)

    print(f"\nFILE: {filename}")
    print("-" * 70)
    print("Shape:", df_temp.shape)

    print("\nColumns:")
    print(list(df_temp.columns))

    print("\nFirst 3 rows:")
    display(df_temp.head(3))

    print("\nMissing values:")
    print(df_temp.isna().sum().sum())

In [ ]:
import pandas as pd
import numpy as np
import os

dataset_dir = "/content/sensAIfood_Perten"

# Load the three cereal datasets
barley_df = pd.read_csv(
    os.path.join(dataset_dir, "Barley_sensAIfood_Perten.csv")
)

corn_df = pd.read_csv(
    os.path.join(dataset_dir, "Corn_sensAIfood_Perten.csv")
)

wheat_df = pd.read_csv(
    os.path.join(dataset_dir, "Wheat_sensAIfood_Perten.csv")
)

# Combine all samples
moisture_df = pd.concat(
    [barley_df, corn_df, wheat_df],
    ignore_index=True
)

# Identify NIR wavelength columns
nir_moisture_cols = [
    col for col in moisture_df.columns
    if str(col).isdigit()
]

# Sort wavelengths numerically
nir_moisture_cols = sorted(
    nir_moisture_cols,
    key=lambda x: int(x)
)

# Prepare X and y
X_moisture = moisture_df[nir_moisture_cols].astype(float).values
y_moisture = moisture_df["Moisture"].astype(float).values

print("MOISTURE DATA PREPARATION")
print("=" * 70)

print("Combined dataset shape:", moisture_df.shape)
print("Number of samples:", len(moisture_df))
print("Number of NIR features:", len(nir_moisture_cols))

print(
    "Wavelength range:",
    nir_moisture_cols[0],
    "to",
    nir_moisture_cols[-1],
    "nm"
)

print("\nTarget: Moisture")
print("Mean:", round(np.mean(y_moisture), 4))
print("Std:", round(np.std(y_moisture), 4))
print("Min:", round(np.min(y_moisture), 4))
print("Median:", round(np.median(y_moisture), 4))
print("Max:", round(np.max(y_moisture), 4))

print("\nSamples by cereal:")
print(moisture_df["Cereal"].value_counts())

print("\nMissing values in X:", np.isnan(X_moisture).sum())
print("Missing values in y:", np.isnan(y_moisture).sum())

print("\nSTATUS: PASS")

In [ ]:
from sklearn.model_selection import train_test_split

# Split the moisture dataset
X_moisture_train, X_moisture_test, y_moisture_train, y_moisture_test = train_test_split(
    X_moisture,
    y_moisture,
    test_size=0.20,
    random_state=42
)

print("MOISTURE TRAIN / TEST SPLIT")
print("=" * 70)

print("Total samples :", len(X_moisture))
print("Training      :", len(X_moisture_train))
print("Testing       :", len(X_moisture_test))
print("NIR features  :", X_moisture_train.shape[1])

print("\nTraining target range:")
print(
    round(y_moisture_train.min(), 4),
    "to",
    round(y_moisture_train.max(), 4)
)

print("Testing target range:")
print(
    round(y_moisture_test.min(), 4),
    "to",
    round(y_moisture_test.max(), 4)
)

print("\nSTATUS: PASS")

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Cross-validation setup
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

component_results = []

# Test a reasonable range of PLS components
max_components = min(40, X_moisture_train.shape[1])

for n_components in range(2, max_components + 1):

    pls_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSRegression(n_components=n_components))
    ])

    scores = cross_val_score(
        pls_pipeline,
        X_moisture_train,
        y_moisture_train,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    rmse = -scores.mean()

    component_results.append({
        "n_components": n_components,
        "CV_RMSE": rmse
    })

# Results table
moisture_pls_results = pd.DataFrame(component_results)

# Best number of components
best_row = moisture_pls_results.loc[
    moisture_pls_results["CV_RMSE"].idxmin()
]

best_moisture_components = int(best_row["n_components"])
best_moisture_cv_rmse = float(best_row["CV_RMSE"])

print("MOISTURE PLS COMPONENT OPTIMIZATION")
print("=" * 70)

print(
    moisture_pls_results
    .sort_values("CV_RMSE")
    .head(10)
    .to_string(index=False)
)

print("\nBest components:", best_moisture_components)
print("Best CV RMSE:", round(best_moisture_cv_rmse, 4))

print("\nSTATUS: PASS")

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Final Moisture PLS model
final_moisture_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pls", PLSRegression(
        n_components=best_moisture_components
    ))
])

# Train only on the training set
final_moisture_model.fit(
    X_moisture_train,
    y_moisture_train
)

# Predict untouched test set
moisture_predictions = (
    final_moisture_model
    .predict(X_moisture_test)
    .ravel()
)

# Evaluation metrics
moisture_mae = mean_absolute_error(
    y_moisture_test,
    moisture_predictions
)

moisture_rmse = np.sqrt(
    mean_squared_error(
        y_moisture_test,
        moisture_predictions
    )
)

moisture_r2 = r2_score(
    y_moisture_test,
    moisture_predictions
)

print("FINAL MOISTURE MODEL")
print("=" * 70)

print("PLS components :", best_moisture_components)
print("Test samples   :", len(y_moisture_test))

print("\nPerformance:")
print("MAE  :", round(moisture_mae, 4))
print("RMSE :", round(moisture_rmse, 4))
print("R²   :", round(moisture_r2, 4))
print("R² % :", round(moisture_r2 * 100, 2), "%")

print("\nSTATUS: MODEL TRAINED")


In [ ]:
import os
import joblib

# Create directory for the Moisture model
moisture_model_dir = "/content/moisture_ai_model"
os.makedirs(moisture_model_dir, exist_ok=True)

# Save trained model
moisture_model_path = os.path.join(
    moisture_model_dir,
    "moisture_pls_model.pkl"
)

joblib.dump(
    final_moisture_model,
    moisture_model_path
)

# Save wavelength information
moisture_wavelengths_path = os.path.join(
    moisture_model_dir,
    "moisture_wavelength_cols.pkl"
)

joblib.dump(
    nir_moisture_cols,
    moisture_wavelengths_path
)

# Save validation metrics
moisture_metrics = {
    "model": "PLSRegression",
    "components": best_moisture_components,
    "test_samples": len(y_moisture_test),
    "MAE": float(moisture_mae),
    "RMSE": float(moisture_rmse),
    "R2": float(moisture_r2),
    "R2_percent": float(moisture_r2 * 100),
    "wavelength_start_nm": int(nir_moisture_cols[0]),
    "wavelength_end_nm": int(nir_moisture_cols[-1]),
    "num_features": len(nir_moisture_cols),
    "dataset_samples": len(moisture_df),
    "dataset_type": "Cereal NIR dataset"
}

metrics_path = os.path.join(
    moisture_model_dir,
    "moisture_model_metrics.pkl"
)

joblib.dump(
    moisture_metrics,
    metrics_path
)

print("MOISTURE MODEL SAVED")
print("=" * 70)
print("✓", moisture_model_path)
print("✓", moisture_wavelengths_path)
print("✓", metrics_path)

print("\nSTATUS: PASS")

In [ ]:
import joblib
import numpy as np

# Reload the saved Moisture model
loaded_moisture_model = joblib.load(
    "/content/moisture_ai_model/moisture_pls_model.pkl"
)

# Test predictions using the reloaded model
original_predictions = final_moisture_model.predict(
    X_moisture_test[:10]
).ravel()

loaded_predictions = loaded_moisture_model.predict(
    X_moisture_test[:10]
).ravel()

# Compare predictions
max_difference = np.max(
    np.abs(original_predictions - loaded_predictions)
)

print("SAVED MOISTURE MODEL VALIDATION")
print("=" * 70)

print("Original model prediction [1]:",
      round(float(original_predictions[0]), 6))

print("Reloaded model prediction [1]:",
      round(float(loaded_predictions[0]), 6))

print("\nMaximum prediction difference:",
      round(float(max_difference), 10))

if max_difference < 1e-10:
    print("\nSTATUS: PASS")
    print("Saved and reloaded model produce identical predictions.")
else:
    print("\nSTATUS: FAIL")
    print("Prediction mismatch detected.")

In [ ]:
# Urea adulteration dataset requirements

urea_dataset_requirements = {
    "input": "NIR spectral measurements",
    "target": "Urea adulteration",
    "minimum_classes": [
        "NORMAL",
        "UREA_ADULTERATED"
    ],
    "preferred_target": "Urea concentration (%)",
    "preferred_sample_design": (
        "Authentic feed samples with known additions of urea "
        "at controlled concentrations"
    ),
    "required_for_training": [
        "NIR spectra",
        "known urea concentration or adulteration label"
    ],
    "model_options": [
        "Classification",
        "Regression",
        "Classification + concentration regression"
    ],
    "scientific_requirement": (
        "Do not create synthetic adulteration labels from CP/NDF/ADF/IVDMD."
    )
}

print("UREA ADULTERATION DATASET REQUIREMENTS")
print("=" * 70)

for key, value in urea_dataset_requirements.items():
    print(f"\n{key}:")
    print(value)

print("\nSTATUS: REQUIREMENTS DEFINED")

In [ ]:
# Urea adulteration module status

urea_module_status = {
    "module": "Urea Adulteration Detection",
    "status": "DATA_REQUIRED",
    "current_training_data": False,
    "preferred_input": "NIR spectrum",
    "preferred_target": "Urea concentration (%)",
    "classification_target": [
        "NORMAL",
        "UREA_ADULTERATED"
    ],
    "training_design": (
        "Authentic cattle feed samples with controlled urea additions "
        "and corresponding NIR spectra"
    ),
    "current_action": (
        "Do not train a Urea model until experimentally labelled or "
        "validated public data are available."
    )
}

print("UREA ADULTERATION MODULE")
print("=" * 70)

for key, value in urea_module_status.items():
    print(f"{key}: {value}")

print("\nSTATUS: DATA REQUIRED")
print("No synthetic Urea labels will be created.")

In [ ]:
# Sand / Silica contamination module requirements

silica_dataset_requirements = {
    "module": "Sand / Silica Contamination Detection",
    "input": "NIR spectral measurements",
    "primary_target": "Silica or mineral matter concentration",
    "alternative_target": "Contaminated / Non-contaminated",
    "preferred_classes": [
        "NORMAL",
        "SAND_SILICA_CONTAMINATED"
    ],
    "preferred_sample_design": (
        "Authentic feed samples with known sand/silica additions "
        "at controlled concentrations"
    ),
    "required_for_quantitative_model": [
        "NIR spectra",
        "laboratory silica or mineral-matter measurement"
    ],
    "required_for_classification": [
        "NIR spectra",
        "validated contamination labels"
    ],
    "model_options": [
        "PLS regression for silica concentration",
        "PLS-DA / SVM classification",
        "Anomaly detection as screening support"
    ],
    "scientific_requirement": (
        "Do not infer silica concentration or contamination labels "
        "from CP/NDF/ADF/IVDMD."
    )
}

print("SAND / SILICA CONTAMINATION")
print("=" * 70)

for key, value in silica_dataset_requirements.items():
    print(f"\n{key}:")
    print(value)

print("\nSTATUS: REQUIREMENTS DEFINED")

In [ ]:
# Sand / Silica module status

silica_module_status = {
    "module": "Sand / Silica Contamination Detection",
    "quantitative_model": "DATA_REQUIRED",
    "classification_model": "DATA_REQUIRED",
    "anomaly_screening": "AVAILABLE",
    "synthetic_labels_created": False,

    "available_capability": {
        "spectral_anomaly_detection": True,
        "specific_silica_detection": False,
        "silica_concentration_prediction": False
    },

    "required_future_data": [
        "NIR spectra of authentic feed",
        "NIR spectra with controlled sand/silica additions",
        "laboratory silica or mineral-matter reference values",
        "validated contaminated/non-contaminated labels"
    ],

    "scientific_note": (
        "The current anomaly detector identifies unusual spectra, "
        "but cannot determine whether the cause is sand, silica, "
        "adulteration, natural variation, maturity, or measurement variation."
    )
}

print("SAND / SILICA MODULE STATUS")
print("=" * 70)

for key, value in silica_module_status.items():
    print(f"\n{key}:")
    print(value)

print("\nSTATUS: DATA REQUIRED")
print("Existing anomaly detector will remain screening support only.")

## 4. Silica Dataset for ML Model training

public DataverseNO replication dataset

In [ ]:
import requests
import json

print("SILICON NIR DATASET - API CHECK")
print("=" * 70)

persistent_id = "doi:10.18710/4PZFHQ"

metadata_url = (
    "https://dataverse.no/api/datasets/:persistentId/"
    f"?persistentId={persistent_id}"
)

response = requests.get(metadata_url, timeout=30)

print("HTTP STATUS:", response.status_code)

if not response.ok:
    raise RuntimeError(
        f"Dataverse request failed: HTTP {response.status_code}"
    )

api_response = response.json()

print("API status:", api_response.get("status"))

# Dataverse API stores the dataset information inside "data"
dataset_data = api_response.get("data")

if dataset_data is None:
    raise RuntimeError(
        "Dataset information was not found inside the API response."
    )

print("\nTop-level data keys:")
print(list(dataset_data.keys()))

# Get the current/latest dataset version
latest_version = dataset_data.get("latestVersion")

if latest_version is None:
    raise RuntimeError(
        "Could not locate latestVersion in the Dataverse response."
    )

# Dataset title
citation_block = (
    latest_version
    .get("metadataBlocks", {})
    .get("citation", {})
)

citation_fields = citation_block.get("fields", [])

dataset_title = None

for field in citation_fields:
    if field.get("typeName") == "title":
        dataset_title = field.get("value")
        break

print("\nDataset title:")
print(dataset_title)

print("\nDataset version:")
print(latest_version.get("versionNumber"))

# Find spectral data file
spectral_file = None

for file_info in latest_version.get("files", []):
    data_file = file_info.get("dataFile", {})
    filename = data_file.get("filename", "")

    print(
        "FILE:",
        filename,
        "| ID:",
        data_file.get("id"),
        "| SIZE:",
        data_file.get("filesize")
    )

    if filename.lower() == "spectral_data.txt":
        spectral_file = data_file

if spectral_file is None:
    raise RuntimeError(
        "Spectral_data.txt was not found in the dataset."
    )

si_file_id = spectral_file["id"]

print("\n" + "=" * 70)
print("TARGET FILE FOUND")
print("Filename :", spectral_file["filename"])
print("File ID  :", si_file_id)
print("Size     :", spectral_file["filesize"], "bytes")

print("\nSTATUS: PASS")

In [ ]:
import requests
import os

si_file_id = 82475

download_url = (
    f"https://dataverse.no/api/access/datafile/{si_file_id}"
)

si_path = "/content/Spectral_data.txt"

print("DOWNLOADING SILICON NIR DATA")
print("=" * 70)

response = requests.get(
    download_url,
    timeout=120
)

print("HTTP STATUS:", response.status_code)

if not response.ok:
    raise RuntimeError(
        f"Download failed with HTTP {response.status_code}"
    )

with open(si_path, "wb") as f:
    f.write(response.content)

print("Downloaded to:", si_path)
print("File size:", os.path.getsize(si_path), "bytes")

print("\nSTATUS: PASS")

In [ ]:
import pandas as pd
import os

si_path = "/content/Spectral_data.txt"

print("SILICON NIR DATASET INSPECTION")
print("=" * 70)

# Show the first 20 raw lines
print("\nFIRST 20 RAW LINES")
print("-" * 70)

with open(si_path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(20):
        line = f.readline()

        if not line:
            break

        print(f"{i+1}: {line.rstrip()}")

# Try common delimiters and inspect the resulting structure
print("\nSTRUCTURE TEST")
print("-" * 70)

for delimiter_name, delimiter in [
    ("TAB", "\t"),
    ("COMMA", ","),
    ("SEMICOLON", ";")
]:

    try:
        test_df = pd.read_csv(
            si_path,
            sep=delimiter,
            nrows=5
        )

        print(
            f"\n{delimiter_name}: "
            f"shape={test_df.shape}, "
            f"columns={list(test_df.columns[:10])}"
        )

    except Exception as e:
        print(f"\n{delimiter_name}: FAILED -> {e}")

print("\nSTATUS: INSPECTION COMPLETE")

In [ ]:
import requests
import os

si_file_id = 82475
si_path = "/content/Spectral_data.txt"

print("RESTORING SILICON NIR DATASET")
print("=" * 70)

download_url = f"https://dataverse.no/api/access/datafile/{si_file_id}"

response = requests.get(
    download_url,
    timeout=120
)

print("HTTP STATUS:", response.status_code)

if not response.ok:
    raise RuntimeError(
        f"Download failed with HTTP {response.status_code}"
    )

with open(si_path, "wb") as f:
    f.write(response.content)

print("File restored:", si_path)
print("File size:", os.path.getsize(si_path), "bytes")

# Verify the file exists before opening it
if not os.path.exists(si_path):
    raise FileNotFoundError(
        "Silicon dataset was not restored."
    )

print("\nFIRST 15 LINES")
print("-" * 70)

with open(
    si_path,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for i in range(15):
        line = f.readline()

        if not line:
            break

        print(f"{i+1}: {line.rstrip()[:500]}")

print("\nSTATUS: RESTORED AND INSPECTED")

In [ ]:
import pandas as pd
import numpy as np

si_path = "/content/Spectral_data.txt"

print("PARSING SILICON NIR DATASET")
print("=" * 70)

# Try whitespace-delimited format first
si_df = pd.read_csv(
    si_path,
    sep=r"\s+",
    engine="python"
)

print("Dataset shape:", si_df.shape)

print("\nCOLUMN NAMES")
print("-" * 70)

print(list(si_df.columns[:30]))

if len(si_df.columns) > 30:
    print("...")
    print("Last 10 columns:")
    print(list(si_df.columns[-10:]))

# Identify numeric columns
numeric_columns = []

for col in si_df.columns:
    converted = pd.to_numeric(
        si_df[col],
        errors="coerce"
    )

    if converted.notna().sum() == len(si_df):
        numeric_columns.append(col)

print("\nNUMERIC COLUMNS:", len(numeric_columns))

print("\nPOSSIBLE REFERENCE / TARGET COLUMNS")
print("-" * 70)

# Display non-spectral-looking columns first
for col in si_df.columns:
    col_str = str(col).lower()

    if not (
        col_str.replace(".", "", 1).replace("-", "", 1).isdigit()
    ):
        print("✓", col)

print("\nFIRST 3 ROWS")
print("-" * 70)

display(si_df.head(3))

print("\nSTATUS: DATASET PARSED")

In [ ]:
import pandas as pd
import numpy as np

si_path = "/content/Spectral_data.txt"

# Reload in case the previous variable was lost
si_df = pd.read_csv(
    si_path,
    sep=r"\s+",
    engine="python"
)

print("SILICON DATASET COLUMN IDENTIFICATION")
print("=" * 70)

print("Shape:", si_df.shape)

# Show ALL column names with their position
print("\nCOLUMN INDEX + NAME")
print("-" * 70)

for i, col in enumerate(si_df.columns):
    print(f"{i:4d} -> {col}")

# Detect columns whose names look like wavelengths
wavelength_columns = []

for col in si_df.columns:
    try:
        value = float(col)

        # Typical NIR wavelength range
        if 400 <= value <= 2500:
            wavelength_columns.append(col)

    except (ValueError, TypeError):
        pass

print("\n" + "=" * 70)
print("DETECTED SPECTRAL COLUMNS")
print("-" * 70)

print("Number of wavelength columns:", len(wavelength_columns))

if wavelength_columns:
    print("First 10:", wavelength_columns[:10])
    print("Last 10 :", wavelength_columns[-10:])
    print(
        "Range   :",
        wavelength_columns[0],
        "to",
        wavelength_columns[-1],
        "nm"
    )

# Non-spectral columns
non_spectral_columns = [
    col for col in si_df.columns
    if col not in wavelength_columns
]

print("\nNON-SPECTRAL COLUMNS")
print("-" * 70)

for col in non_spectral_columns:
    print("✓", col)

# Show first two rows WITHOUT printing the complete spectrum
print("\nFIRST 2 ROWS - NON-SPECTRAL DATA")
print("-" * 70)

display(
    si_df[non_spectral_columns].head(2)
)

print("\nSTATUS: COLUMN IDENTIFICATION COMPLETE")

In [ ]:
import pandas as pd
import numpy as np

si_path = "/content/Spectral_data.txt"

print("SILICON DATA PREPARATION")
print("=" * 70)

# Load dataset
si_df = pd.read_csv(
    si_path,
    sep=r"\s+",
    engine="python"
)

# Remove quotation marks from column names
si_df.columns = (
    si_df.columns
    .astype(str)
    .str.strip()
    .str.strip('"')
    .str.strip("'")
)

print("Cleaned target/reference columns:")
print(
    [
        col for col in si_df.columns
        if not str(col).startswith("X")
    ]
)

# Identify spectral columns X350 ... X2500
si_spectral_cols = []

for col in si_df.columns:
    col_str = str(col)

    if col_str.startswith("X"):
        try:
            wavelength = int(col_str[1:])

            if 350 <= wavelength <= 2500:
                si_spectral_cols.append(col)

        except ValueError:
            pass

# Sort wavelengths numerically
si_spectral_cols = sorted(
    si_spectral_cols,
    key=lambda x: int(str(x)[1:])
)

# Check target
if "Si" not in si_df.columns:
    raise ValueError(
        "Si target column was not found after column-name cleaning."
    )

# Convert to numeric
X_si_full = si_df[si_spectral_cols].apply(
    pd.to_numeric,
    errors="coerce"
).values

y_si_full = pd.to_numeric(
    si_df["Si"],
    errors="coerce"
).values

# Remove invalid rows
valid_mask = (
    np.isfinite(y_si_full)
    & np.isfinite(X_si_full).all(axis=1)
)

X_si = X_si_full[valid_mask]
y_si = y_si_full[valid_mask]

# Sample IDs
if "Sample" in si_df.columns:
    sample_ids_si = (
        si_df.loc[valid_mask, "Sample"]
        .astype(str)
        .values
    )
else:
    sample_ids_si = np.arange(len(y_si)).astype(str)

print("\nDataset shape:", si_df.shape)

print("Valid samples       :", len(X_si))
print("NIR features        :", X_si.shape[1])

print(
    "Wavelength range    :",
    int(str(si_spectral_cols[0])[1:]),
    "to",
    int(str(si_spectral_cols[-1])[1:]),
    "nm"
)

print("\nSilicon target statistics:")
print("Mean   :", round(float(np.mean(y_si)), 4))
print("Std    :", round(float(np.std(y_si)), 4))
print("Min    :", round(float(np.min(y_si)), 4))
print("Median :", round(float(np.median(y_si)), 4))
print("Max    :", round(float(np.max(y_si)), 4))

print("\nFirst 10 Si values:")
print(np.round(y_si[:10], 4))

print("\nInvalid rows removed:", int((~valid_mask).sum()))

print("\nSTATUS: PASS")

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Split Silicon dataset
X_si_train, X_si_test, y_si_train, y_si_test = train_test_split(
    X_si,
    y_si,
    test_size=0.20,
    random_state=42
)

print("SILICON TRAIN / TEST SPLIT")
print("=" * 70)

print("Total samples :", len(X_si))
print("Training      :", len(X_si_train))
print("Testing       :", len(X_si_test))
print("NIR features  :", X_si_train.shape[1])

print("\nTraining Si range:")
print(
    round(float(y_si_train.min()), 4),
    "to",
    round(float(y_si_train.max()), 4)
)

print("Testing Si range:")
print(
    round(float(y_si_test.min()), 4),
    "to",
    round(float(y_si_test.max()), 4)
)

print("\nSTATUS: PASS")

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# 5-fold cross-validation
cv_si = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

si_component_results = []

# Test a practical range of PLS components
max_components_si = min(
    40,
    X_si_train.shape[1],
    X_si_train.shape[0] - 1
)

for n_components in range(2, max_components_si + 1):

    si_pls_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSRegression(
            n_components=n_components
        ))
    ])

    cv_scores = cross_val_score(
        si_pls_pipeline,
        X_si_train,
        y_si_train,
        cv=cv_si,
        scoring="neg_root_mean_squared_error"
    )

    mean_rmse = -cv_scores.mean()

    si_component_results.append({
        "n_components": n_components,
        "CV_RMSE": mean_rmse
    })

# Results table
si_pls_results = pd.DataFrame(si_component_results)

# Select best component count
best_si_row = si_pls_results.loc[
    si_pls_results["CV_RMSE"].idxmin()
]

best_si_components = int(
    best_si_row["n_components"]
)

best_si_cv_rmse = float(
    best_si_row["CV_RMSE"]
)

print("SILICON PLS COMPONENT OPTIMIZATION")
print("=" * 70)

print(
    si_pls_results
    .sort_values("CV_RMSE")
    .head(10)
    .to_string(index=False)
)

print("\nBest components:", best_si_components)
print("Best CV RMSE:", round(best_si_cv_rmse, 4))

print("\nSTATUS: PASS")

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Final Silicon PLS model
final_si_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pls", PLSRegression(
        n_components=best_si_components
    ))
])

# Train on training data only
final_si_model.fit(
    X_si_train,
    y_si_train
)

# Predict untouched test set
si_predictions = (
    final_si_model
    .predict(X_si_test)
    .ravel()
)

# Calculate evaluation metrics
si_mae = mean_absolute_error(
    y_si_test,
    si_predictions
)

si_rmse = np.sqrt(
    mean_squared_error(
        y_si_test,
        si_predictions
    )
)

si_r2 = r2_score(
    y_si_test,
    si_predictions
)

print("FINAL SILICON MODEL")
print("=" * 70)

print("PLS components :", best_si_components)
print("Test samples   :", len(y_si_test))

print("\nPerformance:")
print("MAE  :", round(float(si_mae), 4))
print("RMSE :", round(float(si_rmse), 4))
print("R²   :", round(float(si_r2), 4))
print("R² % :", round(float(si_r2 * 100), 2), "%")

print("\nSTATUS: MODEL TRAINED")

In [17]:
# ------------------------------------------------------------
# SILICON MODEL STATUS
# This model predicts plant/feed silicon concentration.
# It is NOT a sand-adulteration detector.
# ------------------------------------------------------------

ai_target_registry["Sand_Silica"] = {
    "type": "classification_and_regression",
    "status": "PENDING_CORRECT_CALIBRATION_DATA",
    "production_model": None,
    "screening_support": "Spectral anomaly detector",
    "research_reference_model": "final_si_model",
    "research_dataset": "Plant silicon NIR dataset",
    "production_ready": False,
    "reason": (
        "Plant silicon concentration is not equivalent to "
        "sand adulteration in cattle feed."
    )
}

# Explicitly mark the plant-Si model as non-production
si_model_production_status = {
    "model_name": "final_si_model",
    "purpose": "Plant silicon concentration prediction",
    "SIH_sand_detector": False,
    "production_use": False,
    "status": "RESEARCH_ONLY"
}

print("SILICON MODEL STATUS")
print("=" * 70)

for key, value in si_model_production_status.items():
    print(f"{key}: {value}")

print("\nUPDATED SAND/SILICA REGISTRY")
print("=" * 70)

for key, value in ai_target_registry["Sand_Silica"].items():
    print(f"{key}: {value}")

print("\nSTATUS: QUARANTINED")
print("Plant-Si model will NOT be used as the SIH Sand detector.")

NameError: name 'ai_target_registry' is not defined

In [18]:
# ============================================================
# RESTORE AI TARGET REGISTRY
# AND QUARANTINE PLANT-SILICON MODEL
# ============================================================

ai_target_registry = {
    "CP": {
        "type": "regression",
        "status": "TRAINED",
        "production_ready": True,
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },

    "NDF": {
        "type": "regression",
        "status": "TRAINED",
        "production_ready": True,
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },

    "ADF": {
        "type": "regression",
        "status": "TRAINED",
        "production_ready": True,
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },

    "IVDMD": {
        "type": "regression",
        "status": "TRAINED",
        "production_ready": True,
        "source": "Urochloa humidicola NIR + wet chemistry dataset"
    },

    "Moisture": {
        "type": "regression",
        "status": "TRAINED",
        "production_ready": True,
        "source": "sensAIfood Perten cereal NIR dataset"
    },

    "Urea_Adulteration": {
        "type": "classification",
        "status": "PENDING_CORRECT_DATASET",
        "production_ready": False,
        "source": None
    },

    "Sand_Silica": {
        "type": "classification_and_regression",
        "status": "PENDING_CORRECT_CALIBRATION_DATA",
        "production_ready": False,
        "source": None
    },

    "Mineral_Deficiency": {
        "type": "regression_and_classification",
        "status": "PENDING_CORRECT_DATASET",
        "production_ready": False,
        "source": None
    },

    # Plant silicon experiment is deliberately separated
    "Plant_Silicon_Research": {
        "type": "regression",
        "status": "RESEARCH_ONLY",
        "production_ready": False,
        "source": "Plant silicon NIR dataset"
    }
}


# Explicit status for the model we trained earlier
si_model_production_status = {
    "model_name": "final_si_model",
    "purpose": "Plant silicon concentration prediction",
    "SIH_sand_detector": False,
    "production_use": False,
    "status": "RESEARCH_ONLY"
}


# Display status
print("AI TARGET REGISTRY")
print("=" * 80)

for target, info in ai_target_registry.items():
    print(
        f"{target:25} | "
        f"{info['status']:35} | "
        f"Production: {info['production_ready']}"
    )

print("\n" + "=" * 80)
print("PLANT-SILICON MODEL STATUS")
print("=" * 80)

for key, value in si_model_production_status.items():
    print(f"{key}: {value}")

print("\n" + "=" * 80)
print("STATUS: PASS")
print("Plant-Si model is quarantined as RESEARCH_ONLY.")
print("It will NOT be used as the SIH Sand/Silica detector.")

AI TARGET REGISTRY
CP                        | TRAINED                             | Production: True
NDF                       | TRAINED                             | Production: True
ADF                       | TRAINED                             | Production: True
IVDMD                     | TRAINED                             | Production: True
Moisture                  | TRAINED                             | Production: True
Urea_Adulteration         | PENDING_CORRECT_DATASET             | Production: False
Sand_Silica               | PENDING_CORRECT_CALIBRATION_DATA    | Production: False
Mineral_Deficiency        | PENDING_CORRECT_DATASET             | Production: False
Plant_Silicon_Research    | RESEARCH_ONLY                       | Production: False

PLANT-SILICON MODEL STATUS
model_name: final_si_model
purpose: Plant silicon concentration prediction
SIH_sand_detector: False
production_use: False
status: RESEARCH_ONLY

STATUS: PASS
Plant-Si model is quarantined as RESEARCH_ON

In [19]:
# Create a timestamped backup of the current notebook

from google.colab import files
import os
from datetime import datetime

# Current notebook filename
notebook_name = "NIR_Feed_AI_Model.ipynb"

# Verify whether the notebook file is accessible
if os.path.exists(notebook_name):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"NIR_Feed_AI_Model_backup_{timestamp}.ipynb"

    os.rename(notebook_name, backup_name)

    print("BACKUP CREATED")
    print("=" * 70)
    print("File:", backup_name)

    files.download(backup_name)
else:
    print("Notebook file is not directly accessible from the runtime.")
    print("Use Colab: File → Download → Download .ipynb")

Notebook file is not directly accessible from the runtime.
Use Colab: File → Download → Download .ipynb


## 5. Visualize NIR spectra

Each line is one sample. The model learns the relationship between these spectral patterns and laboratory chemistry.

In [ ]:
if "df" in globals():
    X_all = df[wavelength_cols].apply(pd.to_numeric, errors="coerce").to_numpy()
    valid = np.isfinite(X_all).all(axis=1)

    plt.figure(figsize=(12, 5))
    for i in np.where(valid)[0][:20]:
        plt.plot(wavelengths, X_all[i], alpha=0.5)
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Absorbance")
    plt.title("Example NIR spectra")
    plt.tight_layout()
    plt.show()

## 6. First real model: Crude Protein (CP)

We start with **PLS Regression**.

Why PLS?

NIR has hundreds/thousands of highly correlated wavelength variables. PLS is a classic chemometric method designed for this kind of high-dimensional correlated data.

We use **SNV** first. SNV reduces multiplicative/scatter effects by normalizing each individual spectrum.

In [ ]:
if "df" in globals() and "CP" in detect_targets(df):
    cp_result = train_target_model(
        df,
        target_name="CP",
        n_components=10,
        preprocessing="snv",
        test_size=0.20,
        random_state=42,
    )

    print("CP model metrics")
    for k, v in cp_result["metrics"].items():
        print(f"{k}: {v:.4f}")

## 7. Plot measured vs predicted CP

Do not call R² “accuracy”. For regression, report **MAE, RMSE and R²**.

In [ ]:
if "cp_result" in globals():
    y_test = cp_result["y_test"]
    pred = cp_result["pred"]

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, pred, alpha=0.7)
    mn = min(y_test.min(), pred.min())
    mx = max(y_test.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], linestyle="--")
    plt.xlabel("Laboratory CP")
    plt.ylabel("Predicted CP")
    plt.title("CP: measured vs predicted")
    plt.tight_layout()
    plt.show()

## 8. Tune the number of PLS components

Do not blindly choose 10 components. Cross-validation helps find a reasonable complexity.

In [ ]:
if "df" in globals() and "CP" in detect_targets(df):
    X_cp, y_cp, wl_cp, cp_col = prepare_target(df, "CP")
    tuning = tune_pls_components(
        X_cp,
        y_cp,
        preprocessing="snv",
        components=(2, 4, 6, 8, 10, 12, 15, 20),
    )
    display(tuning)

    best_components = int(tuning.iloc[0]["n_components"])
    print("Best CV component count:", best_components)

## 9. Train separate models for the available nutritional targets

The first AI engine will have one regression model per target:

- `Model_CP`
- `Model_NDF`
- `Model_ADF`
- `Model_IVDMD`

This is cleaner than forcing all targets into one model at the beginning.

In [ ]:
results = {}

if "df" in globals():
    for target in ["CP", "NDF", "ADF", "IVDMD"]:
        if target in detect_targets(df):
            # Start with 10 components. Replace with target-specific CV tuning later.
            results[target] = train_target_model(
                df,
                target_name=target,
                n_components=10,
                preprocessing="snv",
                test_size=0.20,
                random_state=42,
            )

            print(f"\n{target}")
            print(results[target]["metrics"])

## 10. Create the first model performance table

In [ ]:
if results:
    performance = pd.DataFrame({
        target: result["metrics"]
        for target, result in results.items()
    }).T

    display(performance)

## 11. Save the trained models

These `.joblib` files are what the future FastAPI backend can load.

In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

for target, result in results.items():
    path = MODEL_DIR / f"nir_pls_{target.lower()}_snv.joblib"
    save_model(result["model"], path)
    print("Saved:", path)

## 12. Build a simple prediction function

This is the bridge between the ML model and Person 5's FastAPI backend.

The backend will eventually receive an NIR spectrum and call these models.

In [ ]:
def predict_sample(models, spectrum):
    spectrum = np.asarray(spectrum, dtype=float).reshape(1, -1)
    output = {}

    for target, model in models.items():
        value = float(model.predict(spectrum).ravel()[0])
        output[target] = value

    return output

if results:
    model_dict = {target: r["model"] for target, r in results.items()}
    example_spectrum = results[next(iter(results))]["X_test"][0]

    prediction = predict_sample(model_dict, example_spectrum)
    print(prediction)

## 13. Add anomaly detection

This is a second AI component.

It does **not** say “this sample is toxic”.

It says: **“this spectrum looks unusual compared with the calibration population.”**

That is much safer and scientifically defensible for the first prototype.

In [ ]:
if "df" in globals():
    # Use complete spectral rows only.
    X_spectral = df[wavelength_cols].apply(pd.to_numeric, errors="coerce")
    X_spectral = X_spectral.dropna().to_numpy(dtype=float)

    anomaly_detector = fit_anomaly_detector(
        X_spectral,
        contamination=0.03,
        random_state=42,
    )

    labels = anomaly_detector.predict(X_spectral[:10])
    print("1 = normal, -1 = anomalous")
    print(labels)

## 14. Your AI architecture

### Version 1

```text
                    ┌─────────────────┐
Feed / Forage ─────►│   NIR Spectrum  │
                    └────────┬────────┘
                             │
                             ▼
                    Spectral preprocessing
                         (SNV / SG)
                             │
                             ▼
                    ┌─────────────────┐
                    │   PLS Models    │
                    └────────┬────────┘
                             │
          ┌──────────────────┼──────────────────┐
          ▼                  ▼                  ▼
         CP                 NDF                ADF
          │                  │                  │
          └──────────────────┼──────────────────┘
                             ▼
                           IVDMD
                             │
                             ▼
                  Quality / Risk Engine
                             │
                             ▼
                    Farmer Recommendation
                             │
                             ▼
                         Mobile App

Parallel:
NIR spectrum ──► anomaly detector ──► unusual sample flag
```

### Version 2

Add:

```text
NIR + Temperature + Humidity + pH + Camera
                ↓
        Multimodal AI Engine
                ↓
Nutrition + Storage Risk + Visible Spoilage
                ↓
       Confidence-aware advice
```

## 15. What NOT to claim yet

Do not tell judges:

- “Our cheap camera detects aflatoxin.”
- “Our DHT11 directly measures protein.”
- “Our model is accurate for every cattle feed.”
- “R² is accuracy.”
- “The public Colombian forage dataset proves Indian feed performance.”

Instead say:

> “The current prototype demonstrates NIR-based nutritional prediction using a public laboratory-calibrated forage dataset. The production version will be locally calibrated using Indian feed and silage samples with laboratory reference measurements. Environmental sensors and computer vision provide complementary storage and visible-spoilage signals.”

That wording is scientifically much stronger.

## 16. Your next implementation sequence

**Day 1:** Run this notebook and understand every cell.

**Day 2:** Tune PLS components and compare raw vs SNV.

**Day 3:** Add Savitzky-Golay first/second derivative preprocessing.

**Day 4:** Compare PLS vs SVR vs Random Forest/XGBoost.

**Day 5:** Add group-aware validation using location/year metadata if available.

**Day 6:** Save the best models and expose them through FastAPI.

**Day 7:** Connect Person 3's ESP32/sensors and Person 5's backend.

The final AI system should only move to farmer-facing advice after validation and domain-defined thresholds are established.